In [ ]:
import time

start_time = time.time()
print(f"Start time recorded: {start_time}")

Start time recorded: 1765264610.3596785


In [ ]:
import time
start_time = 1761994433.9436078

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/phonchi/CryoParticleSegment/blob/main/notebook/03_select_hyperparam_for_extraction_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

### CryoParticleSegment

In [ ]:
do = False # @param{type:"boolean"}
if do :
    %pip install torchinfo -qq
    %pip install -U git+https://github.com/qubvel/segmentation_models.pytorch -qq
    %pip install starfile -qq
    %pip install https://github.com/soft-matter/trackpy/archive/master.zip -qq

> #### ⚠ Notice
>
> You need to restart the kernel after the following step.

In [ ]:
if do :
    %pip install pycuda==2024.1
    %pip install "numpy<2.0"
    %pip install mrcfile -qq

## ⭐ Setup
You must run all codes under this category.

### ✅ Directory Settings

In [ ]:
# @title  { display-mode: "form" }

INPUT_IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
IMAGE_DIR = INPUT_IMAGE_DIR
# @markdown ---

use_denoised_as_pariwise = True # @param {type : "boolean"}
dnzd_pw = use_denoised_as_pariwise
DENOISED_IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
# @markdown ---
LABEL_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset/10017/micrographs_ground_np" # @param {type:"string"}
DATASET_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_with_denoise_reference_CRF/10017/unet_eb5_dice_CRF" # @param {type:"string"}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# @title  { display-mode: "form" }
# @markdown Detect whether using folder in Google Drive as **`RESULT DIR`**📁.
import os
if "content" in IMAGE_DIR.split("/")[:3] or "content" in LABEL_DIR.split("/")[:3]:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    !rm -r /content/sample_data
    if not os.path.exists("/content/image_dir"):
        if "content" in IMAGE_DIR.split("/")[:3]:
            !cp -r {IMAGE_DIR} /content/image_dir
            IMAGE_DIR = "/content/image_dir"
        if "content" in LABEL_DIR.split("/")[:3]:
            !cp -r {LABEL_DIR} /content/label_dir
            LABEL_DIR = "/content/label_dir"
  except:
    pass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
rm: cannot remove '/content/sample_data': No such file or directory


In [ ]:
IMAGE_DIR = "/content/image_dir"

In [ ]:
!git clone https://github.com/phonchi/CryoParticleSegment.git

!wget -O /content/CryoParticleSegment/Modeling/convcrf.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/convcrf.py
!wget -O /content/CryoParticleSegment/Modeling/dataset.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/dataset.py
!wget -O /content/CryoParticleSegment/Modeling/model.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/model.py
!wget -O /content/CryoParticleSegment/Modeling/trainer.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/trainer.py

fatal: destination path 'CryoParticleSegment' already exists and is not an empty directory.
--2025-12-09 07:16:53--  https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/convcrf.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 17753 (17K) [text/plain]
Saving to: ‘/content/CryoParticleSegment/Modeling/convcrf.py’

/content/CryoPartic 100%[===================>]  17.34K  --.-KB/s    in 0.001s  

2025-12-09 07:16:53 (20.7 MB/s) - ‘/content/CryoParticleSegment/Modeling/convcrf.py’ saved [17753/17753]

--2025-12-09 07:16:53--  https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/dataset.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.

In [ ]:
import sys
import os

# Adjust the path relative to your current working directory
module_path = os.path.abspath('CryoParticleSegment/Modeling')

# Add to sys.path if it's not already included
if module_path not in sys.path:
    sys.path.append(module_path)

### ✅ Packages Handling

In [ ]:
# @title  { display-mode: "form" }
# @markdown Useful packages.

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

In [ ]:
# @title  { display-mode: "form" }
# @markdown User-defined packages.

from dataset import MicrographDataset, MicrographDatasetEvery, collate_fn

## ⭐ Main

### ✅ Setting

In [ ]:
# @markdown Parameters.

user = True # @param {type:"boolean"}

In [ ]:
# @markdown Parameters.

BATCH = 8
CROP_SIZE = (512, 512)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# @markdown Set seed.

random_state = 42
torch.manual_seed(random_state)
torch.cuda.manual_seed_all(random_state)

### ✅ Dataset

You can provide a [`transforms.CenterCrop(3840)`](https://docs.pytorch.org/vision/master/generated/torchvision.transforms.CenterCrop.html) object to crop out boundary artifacts.


In [ ]:
crop = transforms.CenterCrop(3840)

In [ ]:
train_dir = os.path.join(IMAGE_DIR, 'train')
train_filenames = np.loadtxt(f"{IMAGE_DIR}/train_filenames.txt", dtype=str)
if dnzd_pw == False:
    train_dataset = MicrographDataset(image_dir=train_dir, label_dir=LABEL_DIR, filenames=train_filenames, crop_size=CROP_SIZE, num_patches = 4, crop=crop)
else:
    dnzd_train_dir = os.path.join(DENOISED_IMAGE_DIR, 'train')
    train_dataset = MicrographDataset(image_dir=train_dir, label_dir=LABEL_DIR, denoised_dir = dnzd_train_dir, filenames=train_filenames, crop_size=CROP_SIZE, num_patches = 4, crop=crop)

In [ ]:
val_dir = os.path.join(IMAGE_DIR, 'val')
val_filenames = np.loadtxt(f"{IMAGE_DIR}/val_filenames.txt", dtype=str)
if dnzd_pw == False:
    val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=LABEL_DIR, filenames=val_filenames, crop_size=CROP_SIZE, crop=crop)
else:
    dnzd_val_dir = os.path.join(DENOISED_IMAGE_DIR, 'val')
    val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=LABEL_DIR, denoised_dir = dnzd_val_dir, filenames=val_filenames, crop_size=CROP_SIZE, crop=crop)
val_loader = DataLoader(val_dataset, batch_size=None, shuffle=False, pin_memory=True, collate_fn=collate_fn)

In [ ]:
if not user:
    test_dir = os.path.join(IMAGE_DIR, 'test')
    test_filenames = np.loadtxt(f"{IMAGE_DIR}/test_filenames.txt", dtype=str)
    if dnzd_pw == False:
        test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=LABEL_DIR, filenames=test_filenames, crop_size=CROP_SIZE)
    else:
        dnzd_test_dir = os.path.join(DENOISED_IMAGE_DIR, 'test')
        test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=LABEL_DIR, denoised_dir = dnzd_test_dir, filenames=test_filenames, crop_size=CROP_SIZE)

    test_loader = DataLoader(test_dataset, batch_size=None, shuffle=False, pin_memory=True, collate_fn=collate_fn)

In [ ]:
for i1, i2, i3, i4, i5 in val_loader: #test loader and reconstruct
    print(i3.dtype, i5.dtype)
    print(i3.shape, i5.shape)
    shape = i5.shape
    break

torch.int64 torch.int64
torch.Size([81, 1, 512, 512]) torch.Size([1, 3840, 3840])


## ⭐ Evaluate

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

from torchvision.utils import save_image
import starfile
import pandas as pd
import matplotlib
from PIL import Image
import cv2

def get_basename_with_uid_removed(path):
  return os.path.basename(path).split(sep='_', maxsplit=1)[-1]


def simple_micrograph_preprocessing(micrograph):
  micrograph_copy = micrograph.copy()
  micrograph_copy = (micrograph_copy-micrograph.mean()+2.5*micrograph.std())/5/micrograph.std()
  micrograph_copy[micrograph_copy<0]=0
  micrograph_copy[micrograph_copy>1]=1
  return micrograph_copy


In [ ]:
# @title  { vertical-output: true, display-mode: "form" }
EMPIAR_ID = 10017 # @param {type:"integer"}
RADIUS = 64 # @param {type:"integer"}
# For 10017
BORDER = 128 # @param {type:"integer"}
SIZE = 4096 # @param {type:"integer"}

In [ ]:
!cp {DATASET_DIR}/{EMPIAR_ID}/filtered_val.star .

In [ ]:
y_size = SIZE
labeled_particles = starfile.read(f"filtered_val.star")['particles']
labeled_particles = labeled_particles[['rlnMicrographName', 'rlnCoordinateX', 'rlnCoordinateY']]
labeled_particles.columns = pd.Index(['image_name', 'x_coord', 'y_coord'])
labeled_particles['image_name'] = labeled_particles['image_name'].apply(get_basename_with_uid_removed)
labeled_particles['image_name'] = labeled_particles['image_name'].apply(lambda s: s.split(".")[0])
labeled_particles['y_coord'] = y_size - labeled_particles['y_coord']
labeled_particles

,image_name,x_coord,y_coord
0,Falcon_2012_06_13-03_22_02_0,2169,1426
1,Falcon_2012_06_13-03_22_02_0,2791,1957
2,Falcon_2012_06_13-03_22_02_0,2372,475
3,Falcon_2012_06_13-03_22_02_0,2635,1047
4,Falcon_2012_06_13-03_22_02_0,3560,3965
...,...,...,...
3540,Falcon_2012_06_12-15_14_01_0,1810,1593
3541,Falcon_2012_06_12-15_14_01_0,1178,1780
3542,Falcon_2012_06_12-15_14_01_0,364,1047
3543,Falcon_2012_06_12-15_14_01_0,961,556


In [ ]:
def preprocess_and_crop(micrograph, crop_size=3840):
    processed_micrograph = simple_micrograph_preprocessing(micrograph)
    if crop_size:
        mic_width, mic_height = processed_micrograph.shape[1], processed_micrograph.shape[0]
        start_x, start_y = (mic_width - crop_size) // 2, (mic_height - crop_size) // 2
        end_x, end_y = start_x + crop_size, start_y + crop_size
        return processed_micrograph[start_y:end_y, start_x:end_x]
    else:
        return processed_micrograph

def plot_micrograph_and_labels(ax, micrograph, labels, coords):
    ax.imshow(micrograph, cmap='gray')
    ax.imshow(labels, cmap='gray', alpha=0.5)
    for x, y in coords:
        corrected_x, corrected_y = x, y
        circle = matplotlib.patches.Circle((corrected_x, corrected_y), radius=RADIUS, fill=False, color='r')
        ax.add_patch(circle)

You can specify a `crop_size` in `preprocess_and_crop()` to remove boundary artifacts during preprocessing.

In [ ]:
label_images = np.empty((0, shape[1], shape[2]), dtype=np.uint8)
gts = []

for idx, (test_image, _, _, grid, _) in enumerate(val_dataset):
    # if idx == 6:
    #     break
    name = val_filenames[idx][:-4]
    micrograph = np.load(f"{IMAGE_DIR}/val/{name}.npy")
    label_path = f"{LABEL_DIR}/{name}.png"
    image = Image.open(label_path)
    label_image = np.array(image)

    cropped_micrograph = preprocess_and_crop(micrograph)
    cropped_label_image = preprocess_and_crop(label_image)
    print(cropped_label_image.shape)
    label_images = np.concatenate((label_images, [cropped_label_image]), axis=0)

    locations = labeled_particles[labeled_particles['image_name'] == name]
    _, ax = plt.subplots(figsize=(12, 12))
    coords = locations[['x_coord', 'y_coord']].values - BORDER
    plot_micrograph_and_labels(ax, cropped_micrograph, cropped_label_image, coords)
    plt.show()
    print(len(coords))
    gts.append(coords)
    ##

#filename = f"{os.path.splitext(checkpoint_path)[0]}.png"
#pred_path = os.path.join(RESULT_DIR, "Each_ckpt", filename)
#save_image(pred_image, pred_path)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
label_images.shape

(6, 3840, 3840)

### CV approach

In [ ]:
radius = RADIUS

In [ ]:
from center_finding import normalize, min_rect_circle, eliminate_near

In [ ]:
cv_list_all = []
cv_config = []
e_factor = [2,4,6]
s_factor = [0.6,1,1.4]

for e in e_factor:
    for s in s_factor:
        cv_list = []
        print(f"e_factor: {e}, s_factor: {s}")
        for img in label_images:
            thresh1 = normalize(img)
            kernel_size = int(radius / 4)  # Example ratio
            kernel = np.ones((kernel_size, kernel_size), np.uint8)
            thresh1 = cv2.erode(thresh1, kernel, iterations=1)

            kernel_size = int(radius / e)  # Example ratio
            kernel = np.ones((kernel_size, kernel_size), np.uint8)
            thresh1 = cv2.erode(thresh1, kernel, iterations=1)

            contours, hierarchy = cv2.findContours(thresh1,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
            cont_array = [c for c in contours]

            # Filter out small/large region and find bounding box with center
            contr_min = radius**s
            c_ = np.array([cv2.contourArea(contour) for contour in contours])
            aa = (c_>contr_min) & (c_<500000)
            aa = aa.tolist()
            c_full_list = [d for d, keep in zip(cont_array, aa) if keep]
            c_list = (list(map(lambda x: min_rect_circle(x, radius), c_full_list)))
            c_list = [x for x in c_list if x is not None]
            cv_list.append(c_list)
            print(len(c_), len(c_list))
        cv_list_all.append(cv_list)
        cv_config.append((e,s))

e_factor: 2, s_factor: 0.6
659 519
669 503
745 591
608 472
652 511
371 302
e_factor: 2, s_factor: 1
659 409
669 391
745 483
608 383
652 379
371 252
e_factor: 2, s_factor: 1.4
659 136
669 131
745 166
608 147
652 100
371 92
e_factor: 4, s_factor: 0.6
568 561
587 579
594 572
522 519
551 545
326 323
e_factor: 4, s_factor: 1
568 561
587 579
594 572
522 519
551 545
326 323
e_factor: 4, s_factor: 1.4
568 561
587 579
594 572
522 519
551 545
326 323
e_factor: 6, s_factor: 0.6
548 532
560 543
547 503
503 492
529 510
320 315
e_factor: 6, s_factor: 1
548 532
560 543
547 503
503 492
529 510
320 315
e_factor: 6, s_factor: 1.4
548 532
560 543
547 503
503 492
529 510
320 315


In [ ]:
observe_id = 0 # @param {type:"integer"}

In [ ]:
for idx, (test_image, _, _, grid, _) in enumerate(val_dataset):
    name = val_filenames[idx][:-4]
    micrograph = np.load(f"{IMAGE_DIR}/val/{name}.npy")
    label_path = f"{LABEL_DIR}/{name}.png"
    image = Image.open(label_path)
    label_image = np.array(image)

    cropped_micrograph = preprocess_and_crop(micrograph)
    cropped_label_image = preprocess_and_crop(label_image)

    #label_images = np.concatenate((label_images, [cropped_label_image]), axis=0)

    #locations = labeled_particles[labeled_particles['image_name'] == name]
    #adjusted_c_list = [(x + 128, y + 128) for x, y in cv_list_all[0][idx]]
    _, ax = plt.subplots(figsize=(12, 12))
    plot_micrograph_and_labels(ax, cropped_micrograph, cropped_label_image, cv_list_all[observe_id][idx])
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

### TrackPy

More information about the Crocker–Grier centroid-finding algorithm is available in the [TrackPy documentation](https://soft-matter.github.io/trackpy/dev/generated/trackpy.locate.html).


In [ ]:
import trackpy as tp

In [ ]:
TrackParticleSize = RADIUS*2-1
curr_adpmass = 0
sep = [0.4, 0.7, 1]
#topn = 500
scale = [0.15, 0.25, 0.35]  # Scale factor (0.5 means reducing the size to half)

In [ ]:
label_images.shape

(6, 3840, 3840)

In [ ]:
tp_list_all = []
tp_config = []
for e in scale:
    for s in sep:
        print(f"e_factor: {e}, s_factor: {s}")
        tp_list = []
        for img in label_images:
            output_image = img
            small_image = cv2.resize(output_image, None, fx=e, fy=e, interpolation=cv2.INTER_AREA)
            # Adjust parameters based on the scale
            sep_s = round(s * TrackParticleSize)+1
            small_sep = int(sep_s * e)
            small_diameter = int(TrackParticleSize * e)
            # Ensure small_diameter is odd
            if small_diameter % 2 == 0:
                small_diameter += 1
            coorTrack = tp.locate(small_image, diameter=small_diameter, minmass=curr_adpmass, separation=small_sep)
            #coorTrack.loc[:,'prob']=[output_image[getattr(coor,'y'),getattr(coor,'x')] for coor in np.floor(coorTrack[['x','y']]).astype('int').itertuples()]
            coorTrack['x'] *= (1/e)
            coorTrack['y'] *= (1/e)
            coords = coorTrack[['x', 'y']].values
            tp_list.append(coords)
            print(len(coords))
        tp_list_all.append(tp_list)
        tp_config.append((e,s))

e_factor: 0.15, s_factor: 0.4
547
575
605
502
525
323
e_factor: 0.15, s_factor: 0.7
485
515
513
459
477
299
e_factor: 0.15, s_factor: 1
296
302
299
289
289
231
e_factor: 0.25, s_factor: 0.4
537
566
592
501
515
321
e_factor: 0.25, s_factor: 0.7
466
500
490
447
462
294
e_factor: 0.25, s_factor: 1
267
283
272
269
264
221
e_factor: 0.35, s_factor: 0.4
524
553
575
497
508
319
e_factor: 0.35, s_factor: 0.7
455
483
489
444
451
287
e_factor: 0.35, s_factor: 1
275
288
284
280
276
220


In [ ]:
observe_id = 3 # @param {type:"integer"}

In [ ]:
for idx, (test_image, _, _, grid, _) in enumerate(val_dataset):
    name = val_filenames[idx][:-4]
    micrograph = np.load(f"{IMAGE_DIR}/val/{name}.npy")
    label_path = f"{LABEL_DIR}/{name}.png"
    image = Image.open(label_path)
    label_image = np.array(image)

    cropped_micrograph = preprocess_and_crop(micrograph)
    cropped_label_image = preprocess_and_crop(label_image)

    #label_images = np.concatenate((label_images, [cropped_label_image]), axis=0)

    locations = labeled_particles[labeled_particles['image_name'] == name]
    _, ax = plt.subplots(figsize=(12, 12))
    plot_micrograph_and_labels(ax, cropped_micrograph, cropped_label_image, tp_list_all[observe_id][idx])
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

### Nonmax

In [ ]:
import pycuda.driver as drv
from center_finding import cleanCanList, pad_image, reLev, reNorm, scoreGpu, getMax, getMax3, gaussian_kernel_2d_opencv, reshape

In [ ]:
#ratio=1
pnum=2000 #initial filtering, larger if candidate is more 2000 for 10017, should inverse proportional to pCans. Use 1000 for 10081
pCans=[0.4, 0.5, 0.6] #the grid size smaller will generate more candidate
overlaps=[0, 0.3, 0.6] #allow for overlap, smaller will delete more
psize=RADIUS*2

# Nor affect too much
level=3
nSep=10
nIter=100

In [ ]:
nms_list_all = []
nms_config = []
for e in pCans:
    for s in overlaps:
        pCan=e
        overlap=s
        print(f"e_factor: {e}, s_factor: {s}")
        nms_list = []
        for img in label_images:
            heatArr=normalize(pad_image(img))
            heatArr=reNorm(heatArr, nSep)
            heatArr=reLev(heatArr,level)
            gaus= gaussian_kernel_2d_opencv(kernel_size = psize,sigma = 0)

            # canList,score=scoreGpuLaunch(heatArr,gaus,psize, gsize, nIter, fscore)
            gsize=psize*pCan
            heat=heatArr.astype(np.float32)
            gaus=gaus.astype(np.float32)
            score=np.zeros(heat.shape).astype(np.float32)
            [sizex,sizey]=heat.shape
            sizex = int(sizex)
            sizey = int(sizey)
            psize = int(psize)
            gsize = int(gsize)

            func = scoreGpu

            tx=16
            ty=16
            bx=(sizex-1)//tx+1
            by=(sizey-1)//ty+1
            print('get score gpu',tx, ty, bx, by, gsize, psize)


            func(drv.In(heat), drv.In(gaus), drv.Out(score), np.int32(sizex), np.int32(sizey), np.int32(psize),
                block=(tx, ty, 1), grid=(int(bx), int(by)))

            # # Calculate necessary dimensions and convert all to int
            numx = int((sizex - 1) // gsize + 1)
            numy = int((sizey - 1) // gsize + 1)
            num = numx * numy
            tnum = num * 5

            # # Create the result array
            res = np.zeros(tnum).astype(np.float32)

            # # Block and grid dimensions
            bx = (numx - 1) // tx + 1
            by = (numy - 1) // ty + 1

            # Get the function from the module
            func = getMax

            # Call the function, ensure all parameters are the correct type
            func(drv.In(score), drv.Out(res), np.int32(gsize), np.int32(sizex), np.int32(sizey), np.int32(numx),
                block=(16, 16, 1), grid=(bx, by, 1))

            # Make sure all dimensions and sizes are properly cast to np.int32 to avoid ambiguity
            niter = np.int32(nIter)
            gsize = np.int32(gsize)
            sizex = np.int32(sizex)
            sizey = np.int32(sizey)
            numx = np.int32(numx)
            tnum = np.int32(num)

            print('get Max3', tx, ty, bx, by, niter, 'other : ', gsize, sizex, sizey, numx, tnum)
            func = getMax3
            # Ensuring the correct parameter order and type for the kernel invocation
            func(drv.In(score), drv.InOut(res), gsize, sizex, sizey, numx, tnum, niter,
                    block=(16, 16, 1), grid=(bx, by, 1))

            canList=reshape(res,num)

            print('Number of Particles before:', len(canList))
            if(len(canList)>pnum):
                canList=canList[:pnum]


            canList=cleanCanList(canList,overlap,psize)
            #canList=reCan(canList,ratio)
            print('Number of Particles:', len(canList))
            nms_list.append([(r[1],r[0]) for r in canList])
        nms_list_all.append(nms_list)
        nms_config.append((e,s))
        print("\n")

e_factor: 0.4, s_factor: 0
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles before: 5353
Number of Particles: 263
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles before: 5494
Number of Particles: 271
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles before: 5391
Number of Particles: 264
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles before: 5344
Number of Particles: 251
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles before: 5386
Number of Particles: 279
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles before: 4911
Number of Particles: 219


e_factor: 0.4, s_factor: 0.3
get score gpu 16 1

In [ ]:
observe_id = 2 # @param {type:"integer"}

In [ ]:
for idx, (test_image, _, _, grid, _) in enumerate(val_dataset):
    name = val_filenames[idx][:-4]
    micrograph = np.load(f"{IMAGE_DIR}/val/{name}.npy")
    label_path = f"{LABEL_DIR}/{name}.png"
    image = Image.open(label_path)
    label_image = np.array(image)

    cropped_micrograph = preprocess_and_crop(micrograph)
    cropped_label_image = preprocess_and_crop(label_image)

    #label_images = np.concatenate((label_images, [cropped_label_image]), axis=0)

    locations = labeled_particles[labeled_particles['image_name'] == name]
    _, ax = plt.subplots(figsize=(12, 12))
    plot_micrograph_and_labels(ax, cropped_micrograph, cropped_label_image, nms_list_all[observe_id][idx])
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

### Score

> #### 🗒 Info
> Here, we compute the score based on the validation set. You may choose to rank the algorithms using either the F-score or mAP. Additionally, the parameter $\beta$ in the F-score can be adjusted: values of $\beta > 1$ place greater emphasis on recall over precision, while $\beta < 1$ give more weight to precision.


In [ ]:
from metrics import centers_to_boxes, calculate_iou_torchvision, evaluate_detection_raw_multiple, f_beta_score, calculate_mAP_multiple_images

In [ ]:
# Assign a default confidence score of 1.0 to all predicted boxes
default_score = 1.0
beta = 1 # @param {type:"number"}
F_score = False # @param {type:"boolean"}

In [ ]:
width = RADIUS*2
height = width
cv_scores = []
for i, cv_list in enumerate(cv_list_all):
    print(f"config {cv_config[i]}")
    iou_matrices = []
    gt_full = []
    pred_full = []
    for idx, gt in enumerate(gts):
        # Convert centers to boxes
        gt_boxes = centers_to_boxes(np.array(gt), width, height)
        pred_boxes = centers_to_boxes(np.array(cv_list[idx]), width, height)
        gt_full.append(gt_boxes)
        pred_full.append(pred_boxes)
        # Calculate IoU Matrix
        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    scores = [torch.full((pred_boxes.shape[0],), default_score) for pred_boxes in pred_full]
    # Evaluate detection
    precision, recall = evaluate_detection_raw_multiple(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=beta)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F_Beta:", f_beta)
    # IoU thresholds for mAP calculation
    iou_thresholds = torch.arange(0.5, 1.0, 0.05)
    # Calculate mAP
    mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores, iou_thresholds)
    print(f"Mean Average Precision (mAP): {mAP_value.item()}")
    cv_scores.append((f_beta, mAP_value.item()))

config (2, 0.6)
Precision: 0.9836063981056213
Recall: 0.8057481646537781
F_Beta: 0.8858379065947974
Mean Average Precision (mAP): 0.6729276776313782
config (2, 1)
Precision: 0.9965359568595886
Recall: 0.6492450833320618
F_Beta: 0.7862480543333482
Mean Average Precision (mAP): 0.5398146510124207
config (2, 1.4)
Precision: 0.9908698201179504
Recall: 0.21868638694286346
F_Beta: 0.3582962736702734
Mean Average Precision (mAP): 0.16561666131019592
config (4, 0.6)
Precision: 0.9730115532875061
Recall: 0.8546813130378723
F_Beta: 0.9100159083476289
Mean Average Precision (mAP): 0.7127978801727295
config (4, 1)
Precision: 0.9730115532875061
Recall: 0.8546813130378723
F_Beta: 0.9100159083476289
Mean Average Precision (mAP): 0.7127978801727295
config (4, 1.4)
Precision: 0.9730115532875061
Recall: 0.8546813130378723
F_Beta: 0.9100159083476289
Mean Average Precision (mAP): 0.7127978801727295
config (6, 0.6)
Precision: 0.9611273407936096
Recall: 0.7928531169891357
F_Beta: 0.8689182420367433
Mean Ave

In [ ]:
width = RADIUS*2
height = width
tp_scores = []

for i, tp_list in enumerate(tp_list_all):
    print(f"config {tp_config[i]}")
    iou_matrices = []
    gt_full = []
    pred_full = []
    for idx, gt in enumerate(gts):
        # Convert centers to boxes
        gt_boxes = centers_to_boxes(np.array(gt), width, height)
        pred_boxes = centers_to_boxes(np.array(tp_list[idx]), width, height)
        gt_full.append(gt_boxes)
        pred_full.append(pred_boxes)
        # Calculate IoU Matrix
        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    scores = [torch.full((pred_boxes.shape[0],), default_score) for pred_boxes in pred_full]
    # Evaluate detection
    precision, recall = evaluate_detection_raw_multiple(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=beta)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F_Beta:", f_beta)
    # IoU thresholds for mAP calculation
    iou_thresholds = torch.arange(0.5, 1.0, 0.05)
    # Calculate mAP
    mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores, iou_thresholds)
    print(f"Mean Average Precision (mAP): {mAP_value.item()}")
    tp_scores.append((f_beta, mAP_value.item()))

config (0.15, 0.4)
Precision: 1.0
Recall: 0.8705930709838867
F_Beta: 0.9308203740175043
Mean Average Precision (mAP): 0.6905925273895264
config (0.15, 0.7)
Precision: 1.0
Recall: 0.7808042168617249
F_Beta: 0.8769119136944995
Mean Average Precision (mAP): 0.6263839602470398
config (0.15, 1)
Precision: 1.0
Recall: 0.49452337622642517
F_Beta: 0.661780717642657
Mean Average Precision (mAP): 0.40420618653297424
config (0.25, 0.4)
Precision: 0.9993948936462402
Recall: 0.8582065105438232
F_Beta: 0.9234351378038682
Mean Average Precision (mAP): 0.7275786995887756
config (0.25, 0.7)
Precision: 1.0
Recall: 0.7567188143730164
F_Beta: 0.8615138725466363
Mean Average Precision (mAP): 0.6486287713050842
config (0.25, 1)
Precision: 1.0
Recall: 0.4584906995296478
F_Beta: 0.6287194010596127
Mean Average Precision (mAP): 0.40111279487609863
config (0.35, 0.4)
Precision: 0.9987950325012207
Recall: 0.8430079817771912
F_Beta: 0.9143129618427916
Mean Average Precision (mAP): 0.6960324645042419
config (0.35,

In [ ]:
width = RADIUS*2
height = width
nms_scores = []

for i, nms_list in enumerate(nms_list_all):
    print(f"config {nms_config[i]}")
    iou_matrices = []
    gt_full = []
    pred_full = []
    for idx, gt in enumerate(gts):
        # Convert centers to boxes
        gt_boxes = centers_to_boxes(np.array(gt), width, height)
        pred_boxes = centers_to_boxes(np.array(nms_list[idx]), width, height)
        gt_full.append(gt_boxes)
        pred_full.append(pred_boxes)
        # Calculate IoU Matrix
        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    scores = [torch.full((pred_boxes.shape[0],), default_score) for pred_boxes in pred_full]
    # Evaluate detection
    precision, recall = evaluate_detection_raw_multiple(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=beta)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F_Beta:", f_beta)
    # IoU thresholds for mAP calculation
    iou_thresholds = torch.arange(0.5, 1.0, 0.05)
    # Calculate mAP
    mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores, iou_thresholds)
    print(f"Mean Average Precision (mAP): {mAP_value.item()}")
    nms_scores.append((f_beta, mAP_value.item()))

config (0.4, 0)
Precision: 0.9872263073921204
Recall: 0.44466593861579895
F_Beta: 0.6131549546784801
Mean Average Precision (mAP): 0.2374921441078186
config (0.4, 0.3)
Precision: 0.9969320893287659
Recall: 0.7433553338050842
F_Beta: 0.8516694152848414
Mean Average Precision (mAP): 0.5959174036979675
config (0.4, 0.6)
Precision: 0.997398853302002
Recall: 0.8677528500556946
F_Beta: 0.9280700288743323
Mean Average Precision (mAP): 0.7357895970344543
config (0.5, 0)
Precision: 0.978132426738739
Recall: 0.4718625247478485
F_Beta: 0.6366146805483559
Mean Average Precision (mAP): 0.23299574851989746
config (0.5, 0.3)
Precision: 0.9942886233329773
Recall: 0.8207053542137146
F_Beta: 0.8991963685809697
Mean Average Precision (mAP): 0.612816333770752
config (0.5, 0.6)
Precision: 0.9649383425712585
Recall: 0.9375086426734924
F_Beta: 0.951025750335122
Mean Average Precision (mAP): 0.7358970642089844
config (0.6, 0)
Precision: 0.9479131102561951
Recall: 0.47327932715415955
F_Beta: 0.6313398062265682

---

## DEP

hybrid

### Watershed Mix_prob and Mix_DT

In [ ]:
import cv2
import numpy as np
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops

def detect_particles_hybrid_variable(prob_map_input, particle_radius=28,
                                     alpha=0.8, beta=0.2,
                                     peak_threshold_ratio=0.1, min_dist_ratio=0.4,
                                     min_solidity=0.85, max_aspect_ratio=4.0): # changed
    """
    Hybrid Watershed with Shape Filtering (Solidity & Aspect Ratio).
    Returns list of [x, y, score].
    alpha, beta : control the weight,
    peak_thres_hold_ratio : control the Minimum intensity ratio of peaks (say having 0 ~ 255 then we need at least 26).
    min_dist_ratio : control the minimum distance between peaks by multiplying the radius as the threshold
    solidility control the minimum solidity (I might delete this)
    aspect_ratio control the maximum aspect ratio (I might delete this)
    """
    # 1. Robust Normalization
    prob_map = prob_map_input.astype(np.float32)
    if prob_map.max() > 1.0:
        prob_map /= 255.0

    # 2. Binarize
    binary_threshold = 0.5
    binary_mask = (prob_map > binary_threshold).astype(np.uint8)

    if np.sum(binary_mask) == 0:
        binary_mask = (prob_map > 0.05).astype(np.uint8) # Fallback

    if np.sum(binary_mask) == 0:
        return []

    # 3. Hybrid Surface (Alpha * Prob + Beta * Dist)
    distance = ndi.distance_transform_edt(binary_mask)
    dist_max = distance.max() if distance.max() > 0 else 1.0
    dt_norm = distance / dist_max

    hybrid_surface = (alpha * (1.0 - prob_map)) - (beta * dt_norm) # * weight the 2 MAP

    # 4. Find Markers
    min_dist = int(particle_radius * min_dist_ratio) # radius * min_distance ratio
    local_max_coords = peak_local_max(
        -hybrid_surface,
        min_distance=min_dist, # Set the min distance
        labels=binary_mask,
        threshold_abs=peak_threshold_ratio # Minimum intensity of peaks.
    )

    markers = np.zeros(hybrid_surface.shape, dtype=int)
    for i, (r, c) in enumerate(local_max_coords):
        markers[r, c] = i + 1

    # 5. Watershed
    labels = watershed(hybrid_surface, markers, mask=binary_mask)

    # 6. Filter & Extract (Area + Solidity + Aspect Ratio)
    valid_particles = []
    expected_area = np.pi * (particle_radius ** 2)
    min_area = expected_area * 0.3
    max_area = expected_area * 3.5

    regions = regionprops(labels)
    for region in regions:
        # Calculate Aspect Ratio
        if region.minor_axis_length > 0:
            aspect_ratio = region.major_axis_length / region.minor_axis_length
        else:
            aspect_ratio = 100.0 # Invalid

        # Shape Checks (changed)
        is_valid_area = (min_area <= region.area <= max_area)
        is_solid = (region.solidity >= min_solidity)
        is_round_enough = (aspect_ratio <= max_aspect_ratio)

        if is_valid_area and is_solid and is_round_enough:
            cy, cx = region.centroid
            score = prob_map[int(cy), int(cx)]
            valid_particles.append([int(cx), int(cy), float(score)])

    return valid_particles

In [ ]:
# @title Hybrid Grid Search: Generate Candidates (with Shape)
watershed_list_all = []
watershed_config = []

# Search Space
# 1. Base Parameters
threshold_levels_ratios = [0.40]
min_dist_ratios = [0.8]
mix_params = [(0.3, 0.7), (0.5, 0.5), (0.7, 0.3)]

# 2. Shape Parameters (New)
# Solidity: 0.9 is strict (very smooth), 0.8 is loose
solidity_levels = [0.9, 0.95]
# Aspect Ratio: 1.2 is strict (round), 1.5 is loose (oval)
aspect_ratio_levels = []

print(f"Starting Grid Search with Shape Filters...")

# Iterate all combinations
import itertools
# Create a single list of all parameter combinations to avoid deep nesting
param_grid = list(itertools.product(
    mix_params, threshold_levels_ratios, min_dist_ratios, solidity_levels, aspect_ratio_levels
))

for params in param_grid:
    (alpha, beta), thresh_r, dist_r, solid, ar = params

    print(f"Config: A={alpha}, B={beta}, T={thresh_r}, Sol={solid}, AR={ar}")
    watershed_list = []

    for img in label_images:
        particles = detect_particles_hybrid_variable(
            img,
            particle_radius=RADIUS,
            alpha=alpha,
            beta=beta,
            peak_threshold_ratio=thresh_r,
            min_dist_ratio=dist_r,
            min_solidity=solid,       # Pass new param
            max_aspect_ratio=ar       # Pass new param
        )
        watershed_list.append(particles)

    watershed_list_all.append(watershed_list)
    # Store config tuple: (alpha, beta, thresh, dist, solidity, aspect_ratio)
    watershed_config.append((alpha, beta, thresh_r, dist_r, solid, ar))

print(f" -> Finished generating {len(watershed_list_all)} candidates.")

Starting Grid Search with Shape Filters...
Config: A=0.3, B=0.7, T=0.1, Sol=0, AR=10.0
Config: A=0.3, B=0.7, T=0.1, Sol=0.2, AR=10.0
Config: A=0.3, B=0.7, T=0.2, Sol=0, AR=10.0
Config: A=0.3, B=0.7, T=0.2, Sol=0.2, AR=10.0
Config: A=0.5, B=0.5, T=0.1, Sol=0, AR=10.0
Config: A=0.5, B=0.5, T=0.1, Sol=0.2, AR=10.0
Config: A=0.5, B=0.5, T=0.2, Sol=0, AR=10.0
Config: A=0.5, B=0.5, T=0.2, Sol=0.2, AR=10.0
Config: A=0.7, B=0.3, T=0.1, Sol=0, AR=10.0
Config: A=0.7, B=0.3, T=0.1, Sol=0.2, AR=10.0
Config: A=0.7, B=0.3, T=0.2, Sol=0, AR=10.0
Config: A=0.7, B=0.3, T=0.2, Sol=0.2, AR=10.0
 -> Finished generating 12 candidates.


In [ ]:
# @title Visualize Hybrid Results (Random 2)
import random
import matplotlib.pyplot as plt
import numpy as np
import os

# 1. Select 2 Random Indices
num_images = len(label_images)
random.seed(42)
selected_indices = sorted(random.sample(range(num_images), 5))

print(f"Visualizing indices: {selected_indices}")

# 2. Iterate through Configs
for i, ws_list in enumerate(watershed_list_all):
    # Hybrid Config: (alpha, beta, thresh, dist, solid)
    cfg = watershed_config[i]

    # --- ADJUSTED SEPARATOR FOR HYBRID ---
    print(f"\n{'='*20} Param Val: Alpha={cfg[0]}, Beta={cfg[1]}, Thresh={cfg[2]}, Dist={cfg[3]}, Sol={cfg[4]} {'='*20}")

    for idx in selected_indices:
        # Load Background
        try:
            name = val_filenames[idx][:-4]
            micrograph_path = f"{IMAGE_DIR}/val/{name}.npy"
            if os.path.exists(micrograph_path):
                micrograph = np.load(micrograph_path)
                bg_image = preprocess_and_crop(micrograph)
            else:
                bg_image = label_images[idx]
        except:
            bg_image = label_images[idx]

        # Get Predictions
        preds = ws_list[idx]
        coords = [p[:2] for p in preds]

        # Plot
        _, ax = plt.subplots(figsize=(12, 12))
        plot_micrograph_and_labels(ax, bg_image, label_images[idx], coords)
        plt.title(f"Img {idx} | Particles: {len(coords)}")
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# @title Hybrid Grid Search: Evaluate & Score
import torch
import os
import json

# Robust Metrics Helper
def evaluate_detection_robust(iou_matrices, iou_threshold=0.5):
    tp_total = 0; fp_total = 0; fn_total = 0
    for iou_matrix in iou_matrices:
        n_gt, n_pred = iou_matrix.shape
        if n_gt == 0 and n_pred == 0: continue
        if n_gt > 0 and n_pred == 0: fn_total += n_gt; continue
        if n_gt == 0 and n_pred > 0: fp_total += n_pred; continue

        max_vals, _ = torch.max(iou_matrix, dim=1)
        detected_mask = max_vals >= iou_threshold
        tp = torch.sum(detected_mask).item()
        tp_total += tp
        fn_total += (n_gt - tp)
        fp_total += (n_pred - tp)

    epsilon = 1e-6
    precision = tp_total / (tp_total + fp_total + epsilon)
    recall = tp_total / (tp_total + fn_total + epsilon)
    return precision, recall

# Main Eval Loop
width = RADIUS * 2
watershed_scores = []

print("Evaluating...")
for i, ws_list in enumerate(watershed_list_all):
    cfg = watershed_config[i] # (alpha, beta, thresh, dist, solid, ar)

    iou_matrices = []
    gt_full = []
    pred_full = []
    scores_list = []

    for idx, gt in enumerate(gts):
        gt_boxes = centers_to_boxes(np.array(gt), width, width)
        gt_full.append(gt_boxes)

        preds = ws_list[idx]
        if len(preds) > 0:
            coords = np.array([p[:2] for p in preds])
            sc = torch.tensor([p[2] for p in preds], dtype=torch.float32)
            pred_boxes = centers_to_boxes(coords, width, width)
        else:
            pred_boxes = torch.empty((0, 4))
            sc = torch.tensor([], dtype=torch.float32)

        pred_full.append(pred_boxes)
        scores_list.append(sc)

        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    # 1. F-Beta
    precision, recall = evaluate_detection_robust(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=0.5)

    # 2. mAP
    iou_thresholds = torch.arange(0.5, 1.0, 0.05)
    try:
        mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores_list, iou_thresholds)
        map_val = mAP_value.item()
    except:
        map_val = 0.0

    print(f"Cfg: Sol={cfg[4]}, AR={cfg[5]} | F1: {f_beta:.4f}, mAP: {map_val:.4f}")
    watershed_scores.append((f_beta, map_val))

# Save Best
if len(watershed_scores) > 0:
    best_idx = max(range(len(watershed_scores)), key=lambda i: watershed_scores[i][0])
    best_cfg = watershed_config[best_idx]

    best_res = {
        "alpha": best_cfg[0],
        "beta": best_cfg[1],
        "thresh": best_cfg[2],
        "dist": best_cfg[3],
        "solidity": best_cfg[4],  # Saved
        "aspect_ratio": best_cfg[5], # Saved
        "f_score": watershed_scores[best_idx][0],
        "map": watershed_scores[best_idx][1]
    }

    print("\n--- BEST CONFIGURATION ---")
    print(best_res)

    param_file_path = os.path.join(RESULT_DIR, 'best_watershed_params.json')
    with open(param_file_path, 'w') as f:
        json.dump(best_res, f, indent=4)
    print(f"✅ Saved to: {param_file_path}")

Evaluating...
Cfg: Sol=0, AR=10.0 | F1: 0.7832, mAP: 0.6763
Cfg: Sol=0.2, AR=10.0 | F1: 0.7832, mAP: 0.6763
Cfg: Sol=0, AR=10.0 | F1: 0.7832, mAP: 0.6763
Cfg: Sol=0.2, AR=10.0 | F1: 0.7832, mAP: 0.6763
Cfg: Sol=0, AR=10.0 | F1: 0.7832, mAP: 0.6763
Cfg: Sol=0.2, AR=10.0 | F1: 0.7832, mAP: 0.6763
Cfg: Sol=0, AR=10.0 | F1: 0.7830, mAP: 0.6763
Cfg: Sol=0.2, AR=10.0 | F1: 0.7830, mAP: 0.6763
Cfg: Sol=0, AR=10.0 | F1: 0.7704, mAP: 0.5969
Cfg: Sol=0.2, AR=10.0 | F1: 0.7704, mAP: 0.5969
Cfg: Sol=0, AR=10.0 | F1: 0.3789, mAP: 0.1083
Cfg: Sol=0.2, AR=10.0 | F1: 0.3789, mAP: 0.1083

--- BEST CONFIGURATION ---
{'alpha': 0.3, 'beta': 0.7, 'thresh': 0.1, 'dist': 0.3, 'solidity': 0, 'aspect_ratio': 10.0, 'f_score': 0.7831538834722067, 'map': 0.6762563586235046}
✅ Saved to: /content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_with_denoise_reference_CRF/10017/unet_eb5_dice_CRF/best_watershed_params.json


---

DT only

In [ ]:
def detect_particles_dt_only(prob_map_input, particle_radius=28,
                             peak_threshold_ratio=0.1, min_dist_ratio=0.4,
                             min_solidity=0.85):
    """
    Distance Transform Only Watershed with Strict Boundary Exclusion (3x Radius).
    """
    # 1. Robust Normalization
    prob_map = prob_map_input.astype(np.float32)
    height, width = prob_map.shape[:2]

    if prob_map.max() > 1.0:
        prob_map /= 255.0

    # 2. Binarize
    binary_threshold = 0.5
    binary_mask = (prob_map > binary_threshold).astype(np.uint8)

    if np.sum(binary_mask) == 0:
        binary_mask = (prob_map > 0.05).astype(np.uint8)

    if np.sum(binary_mask) == 0:
        return []

    # 3. Distance Transform
    distance = ndi.distance_transform_edt(binary_mask)
    dist_max = distance.max() if distance.max() > 0 else 1.0
    dt_norm = distance / dist_max
    dt_surface = -dt_norm

    # 4. Find Markers
    min_dist = int(particle_radius * min_dist_ratio)

    # Swapped 'footprint' for 'min_distance' for better circular separation
    local_max_coords = peak_local_max(
        dt_norm,
        min_distance=min_dist,
        labels=binary_mask,
        threshold_abs=peak_threshold_ratio
    )

    markers = np.zeros(distance.shape, dtype=int)
    for i, (r, c) in enumerate(local_max_coords):
        markers[r, c] = i + 1

    # 5. Watershed
    labels = watershed(dt_surface, markers, mask=binary_mask)

    # 6. Filter & Extract
    valid_particles = []
    expected_area = np.pi * (particle_radius ** 2)
    min_area = expected_area * 0.2
    max_area = expected_area * 3.5

    boundary_margin = particle_radius * 1.8

    regions = regionprops(labels)
    for region in regions:
        is_valid_area = (min_area <= region.area <= max_area)
        is_solid = (region.solidity >= min_solidity)

        if is_valid_area and is_solid:
            # region.centroid calculates the moments internally
            # and returns (row, col) -> (y, x)
            cy, cx = region.centroid

            # Boundary Check with 3x Margin
            if (cx < boundary_margin or cx > (width - boundary_margin) or
                cy < boundary_margin or cy > (height - boundary_margin)):
                continue

            score = prob_map[int(cy), int(cx)]
            valid_particles.append([int(cx), int(cy), float(score)])

    return valid_particles

In [ ]:
# @title DT-Only Grid Search: Generate Candidates (No AR)
watershed_list_all = []
watershed_config = []

# Search Space
threshold_levels_ratios = [0.4]
min_dist_ratios = [0.8, 1.0, 1.2]
solidity_levels = [0.95]

print(f"Starting DT-Only Grid Search...")

import itertools
param_grid = list(itertools.product(
    threshold_levels_ratios, min_dist_ratios, solidity_levels
))

for params in param_grid:
    thresh_r, dist_r, solid = params

    print(f"Config: Thresh={thresh_r}, Dist={dist_r}, Sol={solid}")
    watershed_list = []

    for img in label_images:
        particles = detect_particles_dt_only(
            img,
            particle_radius=RADIUS,
            peak_threshold_ratio=thresh_r,
            min_dist_ratio=dist_r,
            min_solidity=solid
        )
        watershed_list.append(particles)

    watershed_list_all.append(watershed_list)
    # Store config: (thresh, dist, solidity)
    watershed_config.append((thresh_r, dist_r, solid))

print(f" -> Finished generating {len(watershed_list_all)} candidates.")

Starting DT-Only Grid Search...
Config: Thresh=0.4, Dist=0.8, Sol=0.95
Config: Thresh=0.4, Dist=1.0, Sol=0.95
Config: Thresh=0.4, Dist=1.2, Sol=0.95
 -> Finished generating 3 candidates.


In [ ]:
# @title Visualize DT-Only Results (Random 2)
import random
import matplotlib.pyplot as plt
import numpy as np
import os

# 1. Select 2 Random Indices (Fixed Seed)
num_images = len(label_images)
random.seed(42)
selected_indices = sorted(random.sample(range(num_images), 5))

print(f"Visualizing indices: {selected_indices}")

# 2. Iterate through Configs
for i, ws_list in enumerate(watershed_list_all):
    # DT-Only Config: (thresh, dist, solid)
    cfg = watershed_config[i]

    # --- ADJUSTED SEPARATOR FOR DT-ONLY ---
    print(f"\n{'='*20} Param Val: Thresh={cfg[0]}, Dist={cfg[1]}, Sol={cfg[2]} {'='*20}")

    for idx in selected_indices:
        # Load Background
        try:
            name = val_filenames[idx][:-4]
            micrograph_path = f"{IMAGE_DIR}/val/{name}.npy"
            if os.path.exists(micrograph_path):
                micrograph = np.load(micrograph_path)
                bg_image = preprocess_and_crop(micrograph)
            else:
                bg_image = label_images[idx]
        except:
            bg_image = label_images[idx]

        # Get Predictions
        preds = ws_list[idx]
        coords = [p[:2] for p in preds]

        # Plot
        _, ax = plt.subplots(figsize=(12, 12))
        plot_micrograph_and_labels(ax, bg_image, label_images[idx], coords)

        # changed) Added (Thresh=..., Dist=..., Sol=...) to the title
        plt.title(f"Img {idx} | Particles: {len(coords)} (Thresh={cfg[0]}, Dist={cfg[1]}, Sol={cfg[2]})")
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# @title DT-Only Grid Search: Evaluate & Score
import torch
import os
import json

# Metrics Helper
def evaluate_detection_robust(iou_matrices, iou_threshold=0.5):
    tp_total = 0; fp_total = 0; fn_total = 0
    for iou_matrix in iou_matrices:
        n_gt, n_pred = iou_matrix.shape
        if n_gt == 0 and n_pred == 0: continue
        if n_gt > 0 and n_pred == 0: fn_total += n_gt; continue
        if n_gt == 0 and n_pred > 0: fp_total += n_pred; continue

        max_vals, _ = torch.max(iou_matrix, dim=1)
        detected_mask = max_vals >= iou_threshold
        tp = torch.sum(detected_mask).item()
        tp_total += tp
        fn_total += (n_gt - tp)
        fp_total += (n_pred - tp)

    epsilon = 1e-6
    precision = tp_total / (tp_total + fp_total + epsilon)
    recall = tp_total / (tp_total + fn_total + epsilon)
    return precision, recall

# Main Eval Loop
width = RADIUS * 2
watershed_scores = []

print("Evaluating DT-Only Results...")

for i, ws_list in enumerate(watershed_list_all):
    cfg = watershed_config[i] # (thresh, dist, solid)

    iou_matrices = []
    gt_full = []
    pred_full = []
    scores_list = []

    for idx, gt in enumerate(gts):
        gt_boxes = centers_to_boxes(np.array(gt), width, width)
        gt_full.append(gt_boxes)

        preds = ws_list[idx]
        if len(preds) > 0:
            coords = np.array([p[:2] for p in preds])
            sc = torch.tensor([p[2] for p in preds], dtype=torch.float32)
            pred_boxes = centers_to_boxes(coords, width, width)
        else:
            pred_boxes = torch.empty((0, 4))
            sc = torch.tensor([], dtype=torch.float32)

        pred_full.append(pred_boxes)
        scores_list.append(sc)

        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    # 1. F-Beta
    precision, recall = evaluate_detection_robust(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=0.5)

    # 2. mAP
    iou_thresholds = torch.arange(0.5, 1.0, 0.05)
    try:
        mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores_list, iou_thresholds)
        map_val = mAP_value.item()
    except:
        map_val = 0.0

    print(f"Cfg: {cfg} | F1: {f_beta:.4f}, mAP: {map_val:.4f}")
    watershed_scores.append((f_beta, map_val))

# Save Best
if len(watershed_scores) > 0:
    best_idx = max(range(len(watershed_scores)), key=lambda i: watershed_scores[i][0])
    best_cfg = watershed_config[best_idx]

    best_res = {
        "method": "dt_only_no_ar",
        "thresh": best_cfg[0],
        "dist": best_cfg[1],
        "solidity": best_cfg[2],
        "f_score": watershed_scores[best_idx][0],
        "map": watershed_scores[best_idx][1]
    }

    print("\n--- BEST CONFIGURATION (DT ONLY - NO AR) ---")
    print(best_res)

    param_file_path = os.path.join(RESULT_DIR, 'best_watershed_params.json')
    with open(param_file_path, 'w') as f:
        json.dump(best_res, f, indent=4)
    print(f"✅ Saved to: {param_file_path}")

Evaluating DT-Only Results...
Cfg: (0.4, 0.8, 0.95) | F1: 0.9326, mAP: 0.7254
Cfg: (0.4, 1.0, 0.95) | F1: 0.8911, mAP: 0.6201
Cfg: (0.4, 1.2, 0.95) | F1: 0.8301, mAP: 0.5012

--- BEST CONFIGURATION (DT ONLY - NO AR) ---
{'method': 'dt_only_no_ar', 'thresh': 0.4, 'dist': 0.8, 'solidity': 0.95, 'f_score': 0.9325923266645421, 'map': 0.725445568561554}
✅ Saved to: /content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_with_denoise_reference_CRF/10017/unet_eb5_dice_CRF/best_watershed_params.json


---
## DT with circularity, Sphericity, Slenderness at scalar for the mean of front 50% 0.65

In [ ]:
import cv2
import numpy as np
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops
import itertools

def calculate_shape_metrics(region):
    """
    Computes Sphericity, Circularity, and Inverse Slenderness.
    """
    area = region.area
    perimeter = region.perimeter

    if perimeter == 0: return 0, 0, 0

    # 1. Circularity (CR): 4*pi*A / P^2
    circularity = (4 * np.pi * area) / (perimeter ** 2)

    # 2. Sphericity (SP): D_eq / D_circumscribed
    # Approximation: D_circumscribed ≈ Major Axis Length
    d_eq = np.sqrt(4 * area / np.pi)
    d_cir = region.major_axis_length
    sphericity = d_eq / d_cir if d_cir > 0 else 0

    # 3. Inverse Slenderness (ISD): Minor / Major
    # We use Inverse (1.0 = Circle, 0.0 = Line) so 'higher is better' for filtering
    major = region.major_axis_length
    minor = region.minor_axis_length
    inv_slenderness = minor / major if major > 0 else 0

    return sphericity, circularity, inv_slenderness

def detect_particles_dt_only(prob_map_input, particle_radius=28,
                             peak_threshold_ratio=0.1, min_dist_ratio=0.4,
                             min_solidity=0.85):
    """
    DT Watershed with Adaptive Shape Thresholding (Top 50% Mean).
    """
    # 1. Robust Normalization
    prob_map = prob_map_input.astype(np.float32)
    height, width = prob_map.shape[:2]

    if prob_map.max() > 1.0:
        prob_map /= 255.0

    # 2. Binarize
    binary_threshold = 0.5
    binary_mask = (prob_map > binary_threshold).astype(np.uint8)

    if np.sum(binary_mask) == 0:
        binary_mask = (prob_map > 0.05).astype(np.uint8)

    if np.sum(binary_mask) == 0:
        return []

    # 3. Distance Transform
    distance = ndi.distance_transform_edt(binary_mask)
    dist_max = distance.max() if distance.max() > 0 else 1.0
    dt_norm = distance / dist_max
    dt_surface = -dt_norm

    # 4. Find Markers
    min_dist = int(particle_radius * min_dist_ratio)

    local_max_coords = peak_local_max(
        dt_norm,
        min_distance=min_dist,
        labels=binary_mask,
        threshold_abs=peak_threshold_ratio
    )

    markers = np.zeros(distance.shape, dtype=int)
    for i, (r, c) in enumerate(local_max_coords):
        markers[r, c] = i + 1

    # 5. Watershed
    labels = watershed(dt_surface, markers, mask=binary_mask)

    # 6. Filter & Extract (Adaptive Phase) # changed)
    expected_area = np.pi * (particle_radius ** 2)
    min_area = expected_area * 0.2
    max_area = expected_area * 3.5
    boundary_margin = particle_radius * 1.8

    regions = regionprops(labels)

    # Store candidates for adaptive analysis
    candidates = []
    metrics_sp = []
    metrics_cr = []
    metrics_isd = []

    for region in regions:
        # Basic Pre-filtering (Area + Solidity)
        is_valid_area = (min_area <= region.area <= max_area)
        is_solid = (region.solidity >= min_solidity)

        if is_valid_area and is_solid:
            # Calculate advanced metrics # changed)
            sp, cr, isd = calculate_shape_metrics(region)

            candidates.append({
                'region': region,
                'sp': sp, 'cr': cr, 'isd': isd
            })
            metrics_sp.append(sp)
            metrics_cr.append(cr)
            metrics_isd.append(isd)

    if not candidates:
        return []

    # 7. Calculate Dynamic Thresholds (Top 50% Mean) # changed)
    def get_adaptive_cutoff(metric_list):
        if not metric_list: return 0.0
        # Sort descending (Higher = More Circular)
        sorted_vals = sorted(metric_list, reverse=True)
        # Take top 50%
        top_half = sorted_vals[:max(1, len(sorted_vals) // 2)]
        mean_val = np.mean(top_half)
        # Threshold is 65% of that mean
        return mean_val * 0.65

    thresh_sp = get_adaptive_cutoff(metrics_sp)
    thresh_cr = get_adaptive_cutoff(metrics_cr)
    thresh_isd = get_adaptive_cutoff(metrics_isd)

    # 8. Final Selection # changed)
    valid_particles = []

    for cand in candidates:
        # Strict intersection: Must pass ALL shape criteria
        if (cand['sp'] >= thresh_sp and
            cand['cr'] >= thresh_cr and
            cand['isd'] >= thresh_isd):

            region = cand['region']
            cy, cx = region.centroid

            # Boundary Check
            if (cx < boundary_margin or cx > (width - boundary_margin) or
                cy < boundary_margin or cy > (height - boundary_margin)):
                continue

            score = prob_map[int(cy), int(cx)]
            valid_particles.append([int(cx), int(cy), float(score)])

    return valid_particles

In [ ]:
# @title DT-Only Grid Search: Generate Candidates (No AR)
watershed_list_all = []
watershed_config = []

# Search Space
threshold_levels_ratios = [0.4]
min_dist_ratios = [0.8, 1.0, 1.2]
solidity_levels = [0.95]

print(f"Starting DT-Only Grid Search with Adaptive Shape Filtering...")

param_grid = list(itertools.product(
    threshold_levels_ratios, min_dist_ratios, solidity_levels
))

for params in param_grid:
    thresh_r, dist_r, solid = params

    print(f"Config: Thresh={thresh_r}, Dist={dist_r}, Sol={solid}")
    watershed_list = []

    for img in label_images:
        particles = detect_particles_dt_only(
            img,
            particle_radius=RADIUS,
            peak_threshold_ratio=thresh_r,
            min_dist_ratio=dist_r,
            min_solidity=solid
        )
        watershed_list.append(particles)

    watershed_list_all.append(watershed_list)
    watershed_config.append((thresh_r, dist_r, solid))

print(f" -> Finished generating {len(watershed_list_all)} candidates.")

Starting DT-Only Grid Search with Adaptive Shape Filtering...
Config: Thresh=0.4, Dist=0.8, Sol=0.95
Config: Thresh=0.4, Dist=1.0, Sol=0.95
Config: Thresh=0.4, Dist=1.2, Sol=0.95
 -> Finished generating 3 candidates.


In [ ]:
# @title Visualize DT-Only Results (Random 2)
import random
import matplotlib.pyplot as plt
import numpy as np
import os

# 1. Select 2 Random Indices (Fixed Seed)
num_images = len(label_images)
random.seed(42)
selected_indices = sorted(random.sample(range(num_images), 5))

print(f"Visualizing indices: {selected_indices}")

# 2. Iterate through Configs
for i, ws_list in enumerate(watershed_list_all):
    # DT-Only Config: (thresh, dist, solid)
    cfg = watershed_config[i]

    # --- ADJUSTED SEPARATOR FOR DT-ONLY ---
    print(f"\n{'='*20} Param Val: Thresh={cfg[0]}, Dist={cfg[1]}, Sol={cfg[2]} {'='*20}")

    for idx in selected_indices:
        # Load Background
        try:
            name = val_filenames[idx][:-4]
            micrograph_path = f"{IMAGE_DIR}/val/{name}.npy"
            if os.path.exists(micrograph_path):
                micrograph = np.load(micrograph_path)
                bg_image = preprocess_and_crop(micrograph)
            else:
                bg_image = label_images[idx]
        except:
            bg_image = label_images[idx]

        # Get Predictions
        preds = ws_list[idx]
        coords = [p[:2] for p in preds]

        # Plot
        _, ax = plt.subplots(figsize=(12, 12))
        plot_micrograph_and_labels(ax, bg_image, label_images[idx], coords)

        # changed) Added (Thresh=..., Dist=..., Sol=...) to the title
        plt.title(f"Img {idx} | Particles: {len(coords)} (Thresh={cfg[0]}, Dist={cfg[1]}, Sol={cfg[2]})")
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# @title DT-Only Grid Search: Evaluate & Score
import torch
import os
import json

# Metrics Helper
def evaluate_detection_robust(iou_matrices, iou_threshold=0.5):
    tp_total = 0; fp_total = 0; fn_total = 0
    for iou_matrix in iou_matrices:
        n_gt, n_pred = iou_matrix.shape
        if n_gt == 0 and n_pred == 0: continue
        if n_gt > 0 and n_pred == 0: fn_total += n_gt; continue
        if n_gt == 0 and n_pred > 0: fp_total += n_pred; continue

        max_vals, _ = torch.max(iou_matrix, dim=1)
        detected_mask = max_vals >= iou_threshold
        tp = torch.sum(detected_mask).item()
        tp_total += tp
        fn_total += (n_gt - tp)
        fp_total += (n_pred - tp)

    epsilon = 1e-6
    precision = tp_total / (tp_total + fp_total + epsilon)
    recall = tp_total / (tp_total + fn_total + epsilon)
    return precision, recall

# Main Eval Loop
width = RADIUS * 2
watershed_scores = []

print("Evaluating DT-Only Results...")

for i, ws_list in enumerate(watershed_list_all):
    cfg = watershed_config[i] # (thresh, dist, solid)

    iou_matrices = []
    gt_full = []
    pred_full = []
    scores_list = []

    for idx, gt in enumerate(gts):
        gt_boxes = centers_to_boxes(np.array(gt), width, width)
        gt_full.append(gt_boxes)

        preds = ws_list[idx]
        if len(preds) > 0:
            coords = np.array([p[:2] for p in preds])
            sc = torch.tensor([p[2] for p in preds], dtype=torch.float32)
            pred_boxes = centers_to_boxes(coords, width, width)
        else:
            pred_boxes = torch.empty((0, 4))
            sc = torch.tensor([], dtype=torch.float32)

        pred_full.append(pred_boxes)
        scores_list.append(sc)

        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    # 1. F-Beta
    precision, recall = evaluate_detection_robust(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=0.5)

    # 2. mAP
    iou_thresholds = torch.arange(0.5, 1.0, 0.05)
    try:
        mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores_list, iou_thresholds)
        map_val = mAP_value.item()
    except:
        map_val = 0.0

    print(f"Cfg: {cfg} | F1: {f_beta:.4f}, mAP: {map_val:.4f}")
    watershed_scores.append((f_beta, map_val))

# Save Best
if len(watershed_scores) > 0:
    best_idx = max(range(len(watershed_scores)), key=lambda i: watershed_scores[i][0])
    best_cfg = watershed_config[best_idx]

    best_res = {
        "method": "dt_only_no_ar",
        "thresh": best_cfg[0],
        "dist": best_cfg[1],
        "solidity": best_cfg[2],
        "f_score": watershed_scores[best_idx][0],
        "map": watershed_scores[best_idx][1]
    }

    print("\n--- BEST CONFIGURATION (DT ONLY - NO AR) ---")
    print(best_res)

    param_file_path = os.path.join(RESULT_DIR, 'best_watershed_params.json')
    with open(param_file_path, 'w') as f:
        json.dump(best_res, f, indent=4)
    print(f"✅ Saved to: {param_file_path}")

Evaluating DT-Only Results...
Cfg: (0.4, 0.8, 0.95) | F1: 0.9312, mAP: 0.7230
Cfg: (0.4, 1.0, 0.95) | F1: 0.8896, mAP: 0.6181
Cfg: (0.4, 1.2, 0.95) | F1: 0.8274, mAP: 0.4978

--- BEST CONFIGURATION (DT ONLY - NO AR) ---
{'method': 'dt_only_no_ar', 'thresh': 0.4, 'dist': 0.8, 'solidity': 0.95, 'f_score': 0.9312279688758982, 'map': 0.7229866981506348}
✅ Saved to: /content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_with_denoise_reference_CRF/10017/unet_eb5_dice_CRF/best_watershed_params.json


---
## dynamic method  DT with circularity, Sphericity, Slenderness

this mehod use the front 50% mean and minus it with the (max-min)*0.5
but max($\mu_{(50\%)} - (max - min)*0.5$, $\mu_{(50\%)} * 0.7$)

In [ ]:
import cv2
import numpy as np
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops
import itertools

def calculate_shape_metrics(region):
    """
    Computes Sphericity, Circularity, and Inverse Slenderness.
    """
    area = region.area
    perimeter = region.perimeter

    if perimeter == 0: return 0, 0, 0

    # 1. Circularity (CR): 4*pi*A / P^2
    circularity = (4 * np.pi * area) / (perimeter ** 2)

    # 2. Sphericity (SP): D_eq / D_circumscribed
    d_eq = np.sqrt(4 * area / np.pi)
    d_cir = region.major_axis_length
    sphericity = d_eq / d_cir if d_cir > 0 else 0

    # 3. Inverse Slenderness (ISD): Minor / Major
    major = region.major_axis_length
    minor = region.minor_axis_length
    inv_slenderness = minor / major if major > 0 else 0

    return sphericity, circularity, inv_slenderness

def detect_particles_dt_only(prob_map_input, particle_radius=28,
                             peak_threshold_ratio=0.1, min_dist_ratio=0.4,
                             min_solidity=0.85):
    """
    DT Watershed with Range-Based Adaptive Thresholding + Safety Floor.
    Formula: max(Mean - Range*0.5, 0.7 * Mean)
    """
    # 1. Robust Normalization
    prob_map = prob_map_input.astype(np.float32)
    height, width = prob_map.shape[:2]

    if prob_map.max() > 1.0:
        prob_map /= 255.0

    # 2. Binarize
    binary_threshold = 0.5
    binary_mask = (prob_map > binary_threshold).astype(np.uint8)

    if np.sum(binary_mask) == 0:
        binary_mask = (prob_map > 0.05).astype(np.uint8)

    if np.sum(binary_mask) == 0:
        return []

    # 3. Distance Transform
    distance = ndi.distance_transform_edt(binary_mask)
    dist_max = distance.max() if distance.max() > 0 else 1.0
    dt_norm = distance / dist_max
    dt_surface = -dt_norm

    # 4. Find Markers
    min_dist = int(particle_radius * min_dist_ratio)

    local_max_coords = peak_local_max(
        dt_norm,
        min_distance=min_dist,
        labels=binary_mask,
        threshold_abs=peak_threshold_ratio
    )

    markers = np.zeros(distance.shape, dtype=int)
    for i, (r, c) in enumerate(local_max_coords):
        markers[r, c] = i + 1

    # 5. Watershed
    labels = watershed(dt_surface, markers, mask=binary_mask)

    # 6. Filter & Extract (Adaptive Phase)
    expected_area = np.pi * (particle_radius ** 2)
    min_area = expected_area * 0.2
    max_area = expected_area * 3.5
    boundary_margin = particle_radius * 1.8

    regions = regionprops(labels)

    # Store candidates
    candidates = []
    metrics_sp = []
    metrics_cr = []
    metrics_isd = []

    for region in regions:
        # Basic Pre-filtering
        is_valid_area = (min_area <= region.area <= max_area)
        is_solid = (region.solidity >= min_solidity)

        if is_valid_area and is_solid:
            sp, cr, isd = calculate_shape_metrics(region)

            candidates.append({
                'region': region,
                'sp': sp, 'cr': cr, 'isd': isd
            })
            metrics_sp.append(sp)
            metrics_cr.append(cr)
            metrics_isd.append(isd)

    if not candidates:
        return []

    # 7. Calculate Dynamic Thresholds (Max Logic) # changed)
    def get_adaptive_cutoff(metric_list):
        if not metric_list: return 0.0

        # Calculate Range
        max_val = np.max(metric_list)
        min_val = np.min(metric_list)
        data_range = max_val - min_val

        # Calculate Mean of Top 50%
        sorted_vals = sorted(metric_list, reverse=True)
        top_half = sorted_vals[:max(1, len(sorted_vals) // 2)]
        mean_top50 = np.mean(top_half)

        # Base Calculation: Mean - (Range * 0.5)
        calculated_thresh = mean_top50 - (data_range * 0.5)

        # Safety Floor: 0.8 * Mean
        floor_thresh = mean_top50 * 0.8

        # Final: Use the stricter (higher) of the two, or softer?
        # "max(current, 0.8*mean)" usually implies taking the higher value (stricter filter)
        # if the range-based calculation drops too low.
        return np.max([calculated_thresh, floor_thresh,  ])

    thresh_sp = get_adaptive_cutoff(metrics_sp)
    thresh_cr = get_adaptive_cutoff(metrics_cr)
    thresh_isd = get_adaptive_cutoff(metrics_isd)

    # 8. Final Selection
    valid_particles = []

    for cand in candidates:
        if (cand['sp'] >= thresh_sp and
            cand['cr'] >= thresh_cr and
            cand['isd'] >= thresh_isd):

            region = cand['region']
            cy, cx = region.centroid

            # Boundary Check
            if (cx < boundary_margin or cx > (width - boundary_margin) or
                cy < boundary_margin or cy > (height - boundary_margin)):
                continue

            score = prob_map[int(cy), int(cx)]
            valid_particles.append([int(cx), int(cy), float(score)])

    return valid_particles

In [ ]:
# @title DT-Only Grid Search: Generate Candidates
watershed_list_all = []
watershed_config = []

# Search Space
threshold_levels_ratios = [0.4]
min_dist_ratios = [0.8, 1.0, 1.2]
solidity_levels = [0.95]

print(f"Starting DT-Only Grid Search with Max-Adaptive Filtering...")

param_grid = list(itertools.product(
    threshold_levels_ratios, min_dist_ratios, solidity_levels
))

for params in param_grid:
    thresh_r, dist_r, solid = params

    print(f"Config: Thresh={thresh_r}, Dist={dist_r}, Sol={solid}")
    watershed_list = []

    for img in label_images:
        particles = detect_particles_dt_only(
            img,
            particle_radius=RADIUS,
            peak_threshold_ratio=thresh_r,
            min_dist_ratio=dist_r,
            min_solidity=solid
        )
        watershed_list.append(particles)

    watershed_list_all.append(watershed_list)
    watershed_config.append((thresh_r, dist_r, solid))

print(f" -> Finished generating {len(watershed_list_all)} candidates.")

Starting DT-Only Grid Search with Max-Adaptive Filtering...
Config: Thresh=0.4, Dist=0.8, Sol=0.95
Config: Thresh=0.4, Dist=1.0, Sol=0.95
Config: Thresh=0.4, Dist=1.2, Sol=0.95
 -> Finished generating 3 candidates.


In [ ]:
# @title Visualize DT-Only Results (Random 2)
import random
import matplotlib.pyplot as plt
import numpy as np
import os

# 1. Select 2 Random Indices (Fixed Seed)
num_images = len(label_images)
random.seed(42)
selected_indices = sorted(random.sample(range(num_images), 5))

print(f"Visualizing indices: {selected_indices}")

# 2. Iterate through Configs
for i, ws_list in enumerate(watershed_list_all):
    # DT-Only Config: (thresh, dist, solid)
    cfg = watershed_config[i]

    # --- ADJUSTED SEPARATOR FOR DT-ONLY ---
    print(f"\n{'='*20} Param Val: Thresh={cfg[0]}, Dist={cfg[1]}, Sol={cfg[2]} {'='*20}")

    for idx in selected_indices:
        # Load Background
        try:
            name = val_filenames[idx][:-4]
            micrograph_path = f"{IMAGE_DIR}/val/{name}.npy"
            if os.path.exists(micrograph_path):
                micrograph = np.load(micrograph_path)
                bg_image = preprocess_and_crop(micrograph)
            else:
                bg_image = label_images[idx]
        except:
            bg_image = label_images[idx]

        # Get Predictions
        preds = ws_list[idx]
        coords = [p[:2] for p in preds]

        # Plot
        _, ax = plt.subplots(figsize=(12, 12))
        plot_micrograph_and_labels(ax, bg_image, label_images[idx], coords)

        # changed) Added (Thresh=..., Dist=..., Sol=...) to the title
        plt.title(f"Img {idx} | Particles: {len(coords)} (Thresh={cfg[0]}, Dist={cfg[1]}, Sol={cfg[2]})")
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# @title DT-Only Grid Search: Evaluate & Score
import torch
import os
import json

# Metrics Helper
def evaluate_detection_robust(iou_matrices, iou_threshold=0.5):
    tp_total = 0; fp_total = 0; fn_total = 0
    for iou_matrix in iou_matrices:
        n_gt, n_pred = iou_matrix.shape
        if n_gt == 0 and n_pred == 0: continue
        if n_gt > 0 and n_pred == 0: fn_total += n_gt; continue
        if n_gt == 0 and n_pred > 0: fp_total += n_pred; continue

        max_vals, _ = torch.max(iou_matrix, dim=1)
        detected_mask = max_vals >= iou_threshold
        tp = torch.sum(detected_mask).item()
        tp_total += tp
        fn_total += (n_gt - tp)
        fp_total += (n_pred - tp)

    epsilon = 1e-6
    precision = tp_total / (tp_total + fp_total + epsilon)
    recall = tp_total / (tp_total + fn_total + epsilon)
    return precision, recall

# Main Eval Loop
width = RADIUS * 2
watershed_scores = []

print("Evaluating DT-Only Results...")

for i, ws_list in enumerate(watershed_list_all):
    cfg = watershed_config[i] # (thresh, dist, solid)

    iou_matrices = []
    gt_full = []
    pred_full = []
    scores_list = []

    for idx, gt in enumerate(gts):
        gt_boxes = centers_to_boxes(np.array(gt), width, width)
        gt_full.append(gt_boxes)

        preds = ws_list[idx]
        if len(preds) > 0:
            coords = np.array([p[:2] for p in preds])
            sc = torch.tensor([p[2] for p in preds], dtype=torch.float32)
            pred_boxes = centers_to_boxes(coords, width, width)
        else:
            pred_boxes = torch.empty((0, 4))
            sc = torch.tensor([], dtype=torch.float32)

        pred_full.append(pred_boxes)
        scores_list.append(sc)

        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    # 1. F-Beta
    precision, recall = evaluate_detection_robust(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=0.5)

    # 2. mAP
    iou_thresholds = torch.arange(0.5, 1.0, 0.05)
    try:
        mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores_list, iou_thresholds)
        map_val = mAP_value.item()
    except:
        map_val = 0.0

    print(f"Cfg: {cfg} | F1: {f_beta:.4f}, mAP: {map_val:.4f}")
    watershed_scores.append((f_beta, map_val))

# Save Best
if len(watershed_scores) > 0:
    best_idx = max(range(len(watershed_scores)), key=lambda i: watershed_scores[i][0])
    best_cfg = watershed_config[best_idx]

    best_res = {
        "method": "dt_only_no_ar",
        "thresh": best_cfg[0],
        "dist": best_cfg[1],
        "solidity": best_cfg[2],
        "f_score": watershed_scores[best_idx][0],
        "map": watershed_scores[best_idx][1]
    }

    print("\n--- BEST CONFIGURATION (DT ONLY - NO AR) ---")
    print(best_res)

    param_file_path = os.path.join(RESULT_DIR, 'best_watershed_params.json')
    with open(param_file_path, 'w') as f:
        json.dump(best_res, f, indent=4)
    print(f"✅ Saved to: {param_file_path}")

Evaluating DT-Only Results...
Cfg: (0.4, 0.8, 0.95) | F1: 0.8912, mAP: 0.6089
Cfg: (0.4, 1.0, 0.95) | F1: 0.8374, mAP: 0.5040
Cfg: (0.4, 1.2, 0.95) | F1: 0.7603, mAP: 0.3926

--- BEST CONFIGURATION (DT ONLY - NO AR) ---
{'method': 'dt_only_no_ar', 'thresh': 0.4, 'dist': 0.8, 'solidity': 0.95, 'f_score': 0.8911652761797857, 'map': 0.6089215874671936}
✅ Saved to: /content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_with_denoise_reference_CRF/10017/unet_eb5_dice_CRF/best_watershed_params.json


---

## prob_weight_with_log

In [ ]:
import cv2
import numpy as np
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops

def detect_particles_log_dt(prob_map_input, particle_radius=28,
                            peak_threshold_ratio=0.1, min_dist_ratio=0.4,
                            min_solidity=0.85):
    """
    Method: Prob * Log(DT + (e-1))
    - Penalizes edges gently (factor ~0.54) instead of killing them (factor 0).
    - Returns list of [x, y, score].
    """
    # 1. Robust Normalization
    prob_map = prob_map_input.astype(np.float32)
    if prob_map.max() > 1.0:
        prob_map /= 255.0

    # 2. Binarize (Base mask)
    binary_threshold = 0.5
    binary_mask = (prob_map > binary_threshold).astype(np.uint8)

    if np.sum(binary_mask) == 0:
        binary_mask = (prob_map > 0.05).astype(np.uint8)

    if np.sum(binary_mask) == 0:
        return []

    # 3. Distance Transform
    distance = ndi.distance_transform_edt(binary_mask)
    dist_max = distance.max() if distance.max() > 0 else 1.0
    dt_norm = distance / dist_max # 0.0 to 1.0

    # 4. Apply Formula: Prob * Log(DT + (e-1))
    # At DT=0 (edge), factor = ln(e-1) ≈ 0.54
    # At DT=1 (center), factor = ln(e) = 1.0
    log_dt = np.log(dt_norm + (np.e - 1))

    hybrid_surface = prob_map * log_dt

    # 5. Find Markers
    min_dist = int(particle_radius * min_dist_ratio)

    # We look for peaks in the POSITIVE hybrid_surface
    local_max_coords = peak_local_max(
        hybrid_surface,
        footprint=np.ones((min_dist, min_dist), dtype=bool),
        labels=binary_mask,
        threshold_abs=peak_threshold_ratio
    )

    markers = np.zeros(hybrid_surface.shape, dtype=int)
    for i, (r, c) in enumerate(local_max_coords):
        markers[r, c] = i + 1

    # 6. Watershed (Invert surface because watershed finds basins)
    labels = watershed(-hybrid_surface, markers, mask=binary_mask)

    # 7. Filter & Extract
    valid_particles = []
    expected_area = np.pi * (particle_radius ** 2)
    min_area = expected_area * 0.15
    max_area = expected_area * 3.5

    regions = regionprops(labels)
    for region in regions:
        # Shape Checks (Solidity Only, as requested)
        is_valid_area = (min_area <= region.area <= max_area)
        is_solid = (region.solidity >= min_solidity)

        if is_valid_area and is_solid:
            cy, cx = region.centroid
            # Use the original probability as the confidence score
            score = prob_map[int(cy), int(cx)]
            valid_particles.append([int(cx), int(cy), float(score)])

    return valid_particles

In [ ]:
# @title Log(DT) Grid Search: Generate Candidates
watershed_list_all = []
watershed_config = []

# Search Space
threshold_levels_ratios = [0.4]
min_dist_ratios = [0.8, 1.0, 1.2]
solidity_levels = [0.9]

print(f"Starting Grid Search (Prob * Log(DT))...")

import itertools
param_grid = list(itertools.product(
    threshold_levels_ratios, min_dist_ratios, solidity_levels
))

for params in param_grid:
    thresh_r, dist_r, solid = params

    print(f"Config: Thresh={thresh_r}, Dist={dist_r}, Sol={solid}")
    watershed_list = []

    for img in label_images:
        particles = detect_particles_log_dt(
            img,
            particle_radius=RADIUS,
            peak_threshold_ratio=thresh_r,
            min_dist_ratio=dist_r,
            min_solidity=solid
        )
        watershed_list.append(particles)

    watershed_list_all.append(watershed_list)
    watershed_config.append((thresh_r, dist_r, solid))

print(f" -> Finished generating {len(watershed_list_all)} candidates.")

Starting Grid Search (Prob * Log(DT))...
Config: Thresh=0.4, Dist=0.4, Sol=0.9
Config: Thresh=0.4, Dist=0.8, Sol=0.9
Config: Thresh=0.4, Dist=1.2, Sol=0.9
Config: Thresh=0.4, Dist=1.5, Sol=0.9
 -> Finished generating 4 candidates.


In [ ]:
# @title Visualize Log(DT) Results on Micrographs
import random
import matplotlib.pyplot as plt
import numpy as np
import os

# 1. Select 5 Random Images (Fixed Seed)
num_images = len(label_images)
random.seed(42)
selected_indices = sorted(random.sample(range(num_images), 5))

print(f"Visualizing indices: {selected_indices}")

# 2. Iterate through Configs
for i, ws_list in enumerate(watershed_list_all):
    # Config tuple: (thresh, dist, solid)
    cfg = watershed_config[i]

    # Separator
    print(f"\n{'='*20} Param Val: Thresh={cfg[0]}, Dist={cfg[1]}, Sol={cfg[2]} {'='*20}")

    for idx in selected_indices:
        # A. Load Original Micrograph (to plot "on the micrograph")
        try:
            name = val_filenames[idx][:-4]
            micrograph_path = f"{IMAGE_DIR}/val/{name}.npy"

            if os.path.exists(micrograph_path):
                micrograph = np.load(micrograph_path)
                # Crop to match the dimensions of your detection
                bg_image = preprocess_and_crop(micrograph)
            else:
                # Fallback to the label image if micrograph file missing
                bg_image = label_images[idx]
        except:
            bg_image = label_images[idx]

        # B. Get Predictions for this Config
        # preds is [x, y, score]
        preds = ws_list[idx]
        coords = [p[:2] for p in preds] # Extract (x, y)

        # C. Plot
        _, ax = plt.subplots(figsize=(12, 12))

        # We pass label_images[idx] as the overlay so you can see the probability map too
        plot_micrograph_and_labels(ax, bg_image, label_images[idx], coords)

        # changed) Added (Thresh=..., Dist=..., Sol=...) to the title
        plt.title(f"Img {idx} | Particles Found: {len(coords)} (Thresh={cfg[0]}, Dist={cfg[1]}, Sol={cfg[2]})")
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# @title Log(DT) Grid Search: Evaluate & Score
import torch
import os
import json

# Metrics Helper
def evaluate_detection_robust(iou_matrices, iou_threshold=0.5):
    tp_total = 0; fp_total = 0; fn_total = 0
    for iou_matrix in iou_matrices:
        n_gt, n_pred = iou_matrix.shape
        if n_gt == 0 and n_pred == 0: continue
        if n_gt > 0 and n_pred == 0: fn_total += n_gt; continue
        if n_gt == 0 and n_pred > 0: fp_total += n_pred; continue

        max_vals, _ = torch.max(iou_matrix, dim=1)
        detected_mask = max_vals >= iou_threshold
        tp = torch.sum(detected_mask).item()
        tp_total += tp
        fn_total += (n_gt - tp)
        fp_total += (n_pred - tp)

    epsilon = 1e-6
    precision = tp_total / (tp_total + fp_total + epsilon)
    recall = tp_total / (tp_total + fn_total + epsilon)
    return precision, recall

# Main Eval Loop
width = RADIUS * 2
watershed_scores = []

print("Evaluating Log(DT) Results...")

for i, ws_list in enumerate(watershed_list_all):
    cfg = watershed_config[i] # (thresh, dist, solid)

    iou_matrices = []
    gt_full = []
    pred_full = []
    scores_list = []

    for idx, gt in enumerate(gts):
        gt_boxes = centers_to_boxes(np.array(gt), width, width)
        gt_full.append(gt_boxes)

        preds = ws_list[idx]
        if len(preds) > 0:
            coords = np.array([p[:2] for p in preds])
            sc = torch.tensor([p[2] for p in preds], dtype=torch.float32)
            pred_boxes = centers_to_boxes(coords, width, width)
        else:
            pred_boxes = torch.empty((0, 4))
            sc = torch.tensor([], dtype=torch.float32)

        pred_full.append(pred_boxes)
        scores_list.append(sc)

        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    # 1. F-Beta
    precision, recall = evaluate_detection_robust(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=0.5)

    # 2. mAP
    iou_thresholds = torch.arange(0.5, 1.0, 0.05)
    try:
        mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores_list, iou_thresholds)
        map_val = mAP_value.item()
    except:
        map_val = 0.0

    print(f"Cfg: {cfg} | F1: {f_beta:.4f}, mAP: {map_val:.4f}")
    watershed_scores.append((f_beta, map_val))

# Save Best
if len(watershed_scores) > 0:
    best_idx = max(range(len(watershed_scores)), key=lambda i: watershed_scores[i][0])
    best_cfg = watershed_config[best_idx]

    best_res = {
        "method": "prob_log_dt",
        "thresh": best_cfg[0],
        "dist": best_cfg[1],
        "solidity": best_cfg[2],
        "f_score": watershed_scores[best_idx][0],
        "map": watershed_scores[best_idx][1]
    }

    print("\n--- BEST CONFIGURATION (Log DT) ---")
    print(best_res)

    param_file_path = os.path.join(RESULT_DIR, 'best_watershed_params.json')
    with open(param_file_path, 'w') as f:
        json.dump(best_res, f, indent=4)
    print(f"✅ Saved to: {param_file_path}")

Evaluating Log(DT) Results...
Cfg: (0.4, 0.4, 0.9) | F1: 0.8609, mAP: 0.7179
Cfg: (0.4, 0.8, 0.9) | F1: 0.9266, mAP: 0.8127
Cfg: (0.4, 1.2, 0.9) | F1: 0.9281, mAP: 0.7968
Cfg: (0.4, 1.5, 0.9) | F1: 0.9219, mAP: 0.7674

--- BEST CONFIGURATION (Log DT) ---
{'method': 'prob_log_dt', 'thresh': 0.4, 'dist': 1.2, 'solidity': 0.9, 'f_score': 0.9281112973182182, 'map': 0.7967925071716309}
✅ Saved to: /content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_with_denoise_reference_CRF/10017/unet_eb5_dice_CRF/best_watershed_params.json


#### depr

---
contour watershed

In [ ]:
import cv2
import numpy as np
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops

def detect_particles_contour_dt_mix(prob_map_input, particle_radius=28,
                                    binary_threshold=0.2,
                                    clump_solidity_thresh=0.85, # If lower, treat as clump
                                    clump_area_thresh_ratio=1.3, # If area > 1.3x expected, treat as clump
                                    min_dist_ratio=0.4,
                                    peak_threshold_ratio=0.1):
    """
    Contour-Watershed-DT Mix:
    1. Finds contours to define regions.
    2. Classifies regions as "Single" or "Clump".
    3. Splits "Clumps" using local DT + Watershed.
    """
    # 1. Normalize
    prob_map = prob_map_input.astype(np.float32)
    if prob_map.max() > 1.0:
        prob_map /= 255.0

    # 2. Binarize & Find Contours
    binary_mask = (prob_map > binary_threshold).astype(np.uint8)
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    valid_particles = []
    expected_area = np.pi * (particle_radius ** 2)
    min_area = expected_area * 0.15
    max_area = expected_area * 5.0 # Allow large clumps

    # Threshold for "Is this a clump?"
    clump_area_thresh = expected_area * clump_area_thresh_ratio

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area or area > max_area:
            continue

        # Calculate Solidity (Area / Convex Hull Area)
        hull = cv2.convexHull(cnt)
        hull_area = cv2.contourArea(hull)
        if hull_area > 0:
            solidity = float(area) / hull_area
        else:
            solidity = 0

        # --- DECISION: Single or Clump? ---
        is_clump = (solidity < clump_solidity_thresh) or (area > clump_area_thresh)

        if not is_clump:
            # Case A: Simple Single Particle -> Use Centroid
            M = cv2.moments(cnt)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                score = prob_map[cy, cx]
                valid_particles.append([cx, cy, float(score)])
        else:
            # Case B: Clump -> Split with DT + Watershed
            # 1. Create local mask for this contour
            mask = np.zeros_like(binary_mask)
            cv2.drawContours(mask, [cnt], -1, 1, thickness=-1)

            # 2. Local Distance Transform
            dist = ndi.distance_transform_edt(mask)
            if dist.max() == 0: continue

            # 3. Find Peaks (Markers)
            min_dist = int(particle_radius * min_dist_ratio)
            # Use lower threshold for split logic to catch faint sub-particles
            local_peaks = peak_local_max(dist, min_distance=min_dist,
                                         labels=mask,
                                         threshold_abs=particle_radius*peak_threshold_ratio)

            # 4. Local Watershed
            markers = np.zeros_like(mask, dtype=int)
            for i, (r, c) in enumerate(local_peaks):
                markers[r, c] = i + 1

            if len(local_peaks) > 0:
                labels = watershed(-dist, markers, mask=mask)

                # 5. Extract Centers from split regions
                sub_regions = regionprops(labels)
                for region in sub_regions:
                    if region.area >= min_area * 0.5: # Allow slightly smaller pieces
                        cy, cx = region.centroid
                        score = prob_map[int(cy), int(cx)]
                        valid_particles.append([int(cx), int(cy), float(score)])
            else:
                 # Fallback: If no peaks found, just use the clump center
                M = cv2.moments(cnt)
                if M["m00"] != 0:
                    cx = int(M["m10"] / M["m00"])
                    cy = int(M["m01"] / M["m00"])
                    score = prob_map[cy, cx]
                    valid_particles.append([cx, cy, float(score)])

    return valid_particles

In [ ]:
# @title Contour-Mix Grid Search: Generate
watershed_list_all = []
watershed_config = []

# Search Space
# 1. Detection Base
mask_thresholds = [0.4]

# 2. Clump Definition
# If solidity < 0.85 OR area > 1.3x -> Treat as clump and split it
clump_solidity_levels = [0.95]
clump_area_ratios = [0.9, 1.2]

print(f"Starting Contour-Mix Grid Search...")

import itertools
param_grid = list(itertools.product(
    mask_thresholds, clump_solidity_levels, clump_area_ratios
))

for params in param_grid:
    mask_th, solid_th, area_r = params

    print(f"Config: Mask={mask_th}, ClumpSol={solid_th}, ClumpArea={area_r}x")
    watershed_list = []

    for img in label_images:
        particles = detect_particles_contour_dt_mix(
            img,
            particle_radius=RADIUS,
            binary_threshold=mask_th,
            clump_solidity_thresh=solid_th,
            clump_area_thresh_ratio=area_r,
            min_dist_ratio=0.4,       # Fixed for simplicity
            peak_threshold_ratio=0.1  # Fixed for simplicity
        )
        watershed_list.append(particles)

    watershed_list_all.append(watershed_list)
    watershed_config.append((mask_th, solid_th, area_r))

print(f" -> Finished generating {len(watershed_list_all)} candidates.")

Starting Contour-Mix Grid Search...
Config: Mask=0.4, ClumpSol=0.95, ClumpArea=0.9x
Config: Mask=0.4, ClumpSol=0.95, ClumpArea=1.2x
 -> Finished generating 2 candidates.


In [ ]:
# @title Visualize Contour-Mix Results (Random 2)
import random
import matplotlib.pyplot as plt
import numpy as np
import os

# 1. Select 2 Random Indices
num_images = len(label_images)
random.seed(42)
selected_indices = sorted(random.sample(range(num_images), 5))

print(f"Visualizing indices: {selected_indices}")

# 2. Iterate through Configs
for i, ws_list in enumerate(watershed_list_all):
    # Contour-Mix Config: (mask, solid, area)
    cfg = watershed_config[i]

    # --- SEPARATOR ---
    print(f"\n{'='*20} Param Val: Mask={cfg[0]}, ClumpSol={cfg[1]}, AreaRatio={cfg[2]}x {'='*20}")

    for idx in selected_indices:
        # Load Background
        try:
            name = val_filenames[idx][:-4]
            micrograph_path = f"{IMAGE_DIR}/val/{name}.npy"
            if os.path.exists(micrograph_path):
                micrograph = np.load(micrograph_path)
                bg_image = preprocess_and_crop(micrograph)
            else:
                bg_image = label_images[idx]
        except:
            bg_image = label_images[idx]

        # Get Predictions
        preds = ws_list[idx]
        coords = [p[:2] for p in preds]

        # Plot
        _, ax = plt.subplots(figsize=(12, 12))
        plot_micrograph_and_labels(ax, bg_image, label_images[idx], coords)
        plt.title(f"Img {idx} | Particles: {len(coords)}")
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# @title Contour-Mix Grid Search: Generate
watershed_list_all = []
watershed_config = []

# Search Space
# 1. Detection Base
mask_thresholds = [0.2, 0.3]

# 2. Clump Definition
# If solidity < 0.85 OR area > 1.3x -> Treat as clump and split it
clump_solidity_levels = [0.85, 0.9]
clump_area_ratios = [1.3, 1.5]

print(f"Starting Contour-Mix Grid Search...")

import itertools
param_grid = list(itertools.product(
    mask_thresholds, clump_solidity_levels, clump_area_ratios
))

for params in param_grid:
    mask_th, solid_th, area_r = params

    print(f"Config: Mask={mask_th}, ClumpSol={solid_th}, ClumpArea={area_r}x")
    watershed_list = []

    for img in label_images:
        particles = detect_particles_contour_dt_mix(
            img,
            particle_radius=RADIUS,
            binary_threshold=mask_th,
            clump_solidity_thresh=solid_th,
            clump_area_thresh_ratio=area_r,
            min_dist_ratio=0.4,       # Fixed for simplicity
            peak_threshold_ratio=0.1  # Fixed for simplicity
        )
        watershed_list.append(particles)

    watershed_list_all.append(watershed_list)
    watershed_config.append((mask_th, solid_th, area_r))

print(f" -> Finished generating {len(watershed_list_all)} candidates.")

Starting Contour-Mix Grid Search...
Config: Mask=0.2, ClumpSol=0.85, ClumpArea=1.3x
Config: Mask=0.2, ClumpSol=0.85, ClumpArea=1.5x
Config: Mask=0.2, ClumpSol=0.9, ClumpArea=1.3x
Config: Mask=0.2, ClumpSol=0.9, ClumpArea=1.5x
Config: Mask=0.3, ClumpSol=0.85, ClumpArea=1.3x
Config: Mask=0.3, ClumpSol=0.85, ClumpArea=1.5x
Config: Mask=0.3, ClumpSol=0.9, ClumpArea=1.3x
Config: Mask=0.3, ClumpSol=0.9, ClumpArea=1.5x
 -> Finished generating 8 candidates.


#### 04

---
DRLSE (Distance Regularize Level Set Evolution) with watershed

In [ ]:
import cv2
import numpy as np
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.measure import regionprops
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# --- 1. DRLSE Core Functions (Li et al. 2010) ---
def drlse_edge_indicator(img, sigma=1.5):
    """
    Computes the edge indicator function g = 1 / (1 + |grad(G*I)|^2)
    """
    img_smooth = ndi.gaussian_filter(img, sigma)
    iy, ix = np.gradient(img_smooth)
    abs_grad_sq = ix**2 + iy**2
    g = 1.0 / (1.0 + abs_grad_sq)
    return g

def drlse_update(phi, g, lambda_param, mu, alpha, epsilon, timestep):
    """
    One iteration of DRLSE evolution.
    """
    # Central difference for gradients
    phi_y, phi_x = np.gradient(phi)
    s = np.sqrt(phi_x**2 + phi_y**2)
    small_num = 1e-10

    # 1. Distance Regularization Term (maintains |grad(phi)| = 1)
    # Potential p(s) = 0.5 * (s-1)^2 -> d_p(s) = (s-1)/s
    nx = phi_x / (s + small_num)
    ny = phi_y / (s + small_num)

    curvature = divergence(nx, ny)
    dist_reg_term = divergence(d_p(s) * nx, d_p(s) * ny) + 4 * del2(phi)

    # 2. Weighted Length Term (smoothness)
    # div(g * grad(phi)/|grad(phi)|)
    dirac_phi = dirac(phi, epsilon)
    area_term = dirac_phi * g # Balloon force (alpha)
    edge_term = dirac_phi * (
        (np.gradient(g)[0] * ny + np.gradient(g)[1] * nx) +
        g * curvature
    )

    # Evolution
    # phi_t = mu * dist_reg + lambda * edge_term + alpha * area_term
    phi_new = phi + timestep * (
        mu * dist_reg_term +
        lambda_param * edge_term +
        alpha * area_term
    )
    return phi_new

# --- DRLSE Helpers ---
def d_p(s):
    return (s > 0).astype(float) * (1 - 1/(s+1e-10)) + (s <= 0).astype(float) # Derivative of potential

def div(nx, ny):
    nxx, _ = np.gradient(nx)
    _, nyy = np.gradient(ny)
    return nxx + nyy

def divergence(fx, fy):
    fxx, _ = np.gradient(fx)
    _, fyy = np.gradient(fy)
    return fxx + fyy

def del2(phi):
    pxx, _ = np.gradient(np.gradient(phi)[0])
    _, pyy = np.gradient(np.gradient(phi)[1])
    return pxx + pyy

def dirac(x, epsilon):
    f = (1/epsilon) * (1 + np.cos(np.pi * x / epsilon))
    f[np.abs(x) > epsilon] = 0
    return 0.5 * f

# --- 2. Smart Filter (Isolation Forest) ---
def filter_particles_smart(particles_raw, contamination=0.1):
    """
    Uses Isolation Forest to find 'average' particle shapes and remove outliers.
    particles_raw: List of [x, y, score, area, solidity, aspect_ratio]
    """
    if len(particles_raw) < 5:
        return particles_raw # Too few to filter

    # Extract Features for Clustering: [Score, Area, Solidity, AspectRatio]
    data = np.array([p[2:] for p in particles_raw])

    # Normalize features (Crucial for ML)
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)

    # Detect Outliers
    # contamination: expected proportion of "junk" (e.g., 0.1 = 10% removal)
    clf = IsolationForest(contamination=contamination, random_state=42)
    preds = clf.fit_predict(data_scaled) # 1 for inlier, -1 for outlier

    valid_particles = []
    for i, pred in enumerate(preds):
        if pred == 1:
            # Keep only x, y, score
            valid_particles.append(particles_raw[i][:3])

    return valid_particles

# --- 3. Main Detection Function ---
def detect_particles_drlse_smart(prob_map_input, particle_radius=28,
                                 drlse_iter=10,
                                 outlier_fraction=0.1):
    """
    Pipeline:
    1. Initial Peaks -> 2. DRLSE Refinement -> 3. Feature Extraction -> 4. Smart Filtering
    """
    # 1. Preprocess
    prob_map = prob_map_input.astype(np.float32)
    if prob_map.max() > 1.0: prob_map /= 255.0

    # Compute Edge Indicator g once
    g = drlse_edge_indicator(prob_map)

    # 2. Initialize Phi (Level Set Function)
    # Using simple threshold as initial binary mask (-2 for outside, 2 for inside)
    # c0 = 2
    binary_mask = (prob_map > 0.2).astype(np.float32)
    phi = binary_mask * 4 - 2 # Map 0/1 to -2/+2

    # 3. Evolve with DRLSE
    # Parameters tuned for typical EM maps
    lambda_param = 5.0 # Weight of edge attraction
    mu = 0.04          # Distance regularization weight
    alpha = -2.0       # Balloon force (negative = expand into high prob areas)
    epsilon = 1.5      # Width of Dirac delta
    timestep = 1.0     # Time step

    for _ in range(drlse_iter):
        phi = drlse_update(phi, g, lambda_param, mu, alpha, epsilon, timestep)

    # 4. Extract Final Shapes
    # Zero-crossing of phi defines the contour.
    # Inside is positive (because we set inside to +2)
    final_mask = (phi > 0).astype(np.uint8)

    # Label and Extract Features
    labels, _ = ndi.label(final_mask)
    regions = regionprops(labels)

    raw_candidates = [] # [x, y, score, area, solidity, ar]

    expected_area = np.pi * (particle_radius ** 2)

    for region in regions:
        # Very rough pre-filter to avoid garbage
        if region.area < expected_area * 0.1: continue

        cy, cx = region.centroid

        # Features
        area_feat = region.area
        solidity_feat = region.solidity
        if region.minor_axis_length > 0:
            ar_feat = region.major_axis_length / region.minor_axis_length
        else:
            ar_feat = 1.0

        score = prob_map[int(cy), int(cx)]

        raw_candidates.append([int(cx), int(cy), float(score), area_feat, solidity_feat, ar_feat])

    # 5. Smart Filtering (Isolation Forest)
    # Automatically finds the "majority cluster" of shapes and rejects the rest
    final_particles = filter_particles_smart(raw_candidates, contamination=outlier_fraction)

    return final_particles

In [ ]:
# @title DRLSE & Smart Filter Grid Search: Generate
watershed_list_all = []
watershed_config = []

# Search Space
# 1. DRLSE Iterations: 0 = No refinement, 20 = Heavy refinement
drlse_iters = [200, 500]

# 2. Smart Filter Strictness
# 0.05 = Remove 5% worst shapes
# 0.20 = Remove 20% worst shapes (keep only the very 'standard' ones)
outlier_fractions = [0.05, 0.1, 0.2]

print(f"Starting DRLSE + Smart Filter Search...")

import itertools
param_grid = list(itertools.product(
    drlse_iters, outlier_fractions
))

for params in param_grid:
    iters, out_frac = params

    print(f"Config: DRLSE_Iter={iters}, OutlierFrac={out_frac}")
    watershed_list = []

    for img in label_images:
        particles = detect_particles_drlse_smart(
            img,
            particle_radius=RADIUS,
            drlse_iter=iters,
            outlier_fraction=out_frac
        )
        watershed_list.append(particles)

    watershed_list_all.append(watershed_list)
    watershed_config.append((iters, out_frac))

print(f" -> Finished generating {len(watershed_list_all)} candidates.")

Starting DRLSE + Smart Filter Search...
Config: DRLSE_Iter=0, OutlierFrac=0.05
Config: DRLSE_Iter=0, OutlierFrac=0.1
Config: DRLSE_Iter=0, OutlierFrac=0.2
Config: DRLSE_Iter=10, OutlierFrac=0.05
Config: DRLSE_Iter=10, OutlierFrac=0.1
Config: DRLSE_Iter=10, OutlierFrac=0.2
Config: DRLSE_Iter=30, OutlierFrac=0.05
Config: DRLSE_Iter=30, OutlierFrac=0.1
Config: DRLSE_Iter=30, OutlierFrac=0.2
 -> Finished generating 9 candidates.


In [ ]:
# @title Visualize DRLSE Results (Random 2)
import random
import matplotlib.pyplot as plt
import numpy as np
import os

# 1. Select 2 Random Indices
num_images = len(label_images)
random.seed(42)
selected_indices = sorted(random.sample(range(num_images), 5))

print(f"Visualizing indices: {selected_indices}")

# 2. Iterate through Configs
for i, ws_list in enumerate(watershed_list_all):
    # DRLSE Config: (iter, outlier_frac)
    cfg = watershed_config[i]

    # --- SEPARATOR ---
    print(f"\n{'='*20} Param Val: DRLSE_Iter={cfg[0]}, OutlierFrac={cfg[1]} {'='*20}")

    for idx in selected_indices:
        # Load Background
        try:
            name = val_filenames[idx][:-4]
            micrograph_path = f"{IMAGE_DIR}/val/{name}.npy"
            if os.path.exists(micrograph_path):
                micrograph = np.load(micrograph_path)
                bg_image = preprocess_and_crop(micrograph)
            else:
                bg_image = label_images[idx]
        except:
            bg_image = label_images[idx]

        # Get Predictions
        preds = ws_list[idx]
        coords = [p[:2] for p in preds]

        # Plot
        _, ax = plt.subplots(figsize=(12, 12))
        plot_micrograph_and_labels(ax, bg_image, label_images[idx], coords)
        plt.title(f"Img {idx} | Particles: {len(coords)}")
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# @title DRLSE & Smart Filter Grid Search: Evaluate
import torch
import os
import json

# Robust Metrics Helper
def evaluate_detection_robust(iou_matrices, iou_threshold=0.5):
    tp_total = 0; fp_total = 0; fn_total = 0
    for iou_matrix in iou_matrices:
        n_gt, n_pred = iou_matrix.shape
        if n_gt == 0 and n_pred == 0: continue
        if n_gt > 0 and n_pred == 0: fn_total += n_gt; continue
        if n_gt == 0 and n_pred > 0: fp_total += n_pred; continue

        max_vals, _ = torch.max(iou_matrix, dim=1)
        detected_mask = max_vals >= iou_threshold
        tp = torch.sum(detected_mask).item()
        tp_total += tp
        fn_total += (n_gt - tp)
        fp_total += (n_pred - tp)

    epsilon = 1e-6
    precision = tp_total / (tp_total + fp_total + epsilon)
    recall = tp_total / (tp_total + fn_total + epsilon)
    return precision, recall

width = RADIUS * 2
watershed_scores = []

print("Evaluating DRLSE Results...")

for i, ws_list in enumerate(watershed_list_all):
    cfg = watershed_config[i] # (iter, outlier_frac)

    iou_matrices = []
    gt_full = []
    pred_full = []
    scores_list = []

    for idx, gt in enumerate(gts):
        gt_boxes = centers_to_boxes(np.array(gt), width, width)
        gt_full.append(gt_boxes)

        preds = ws_list[idx]
        if len(preds) > 0:
            coords = np.array([p[:2] for p in preds])
            sc = torch.tensor([p[2] for p in preds], dtype=torch.float32)
            pred_boxes = centers_to_boxes(coords, width, width)
        else:
            pred_boxes = torch.empty((0, 4))
            sc = torch.tensor([], dtype=torch.float32)

        pred_full.append(pred_boxes)
        scores_list.append(sc)

        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    precision, recall = evaluate_detection_robust(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=0.5)

    try:
        iou_thresholds = torch.arange(0.5, 1.0, 0.05)
        mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores_list, iou_thresholds)
        map_val = mAP_value.item()
    except:
        map_val = 0.0

    print(f"Iter={cfg[0]}, OutFrac={cfg[1]} | F1: {f_beta:.4f}, mAP: {map_val:.4f}")
    watershed_scores.append((f_beta, map_val))

# Save Best
if len(watershed_scores) > 0:
    best_idx = max(range(len(watershed_scores)), key=lambda i: watershed_scores[i][0])
    best_cfg = watershed_config[best_idx]

    best_res = {
        "method": "drlse_smart_filter",
        "drlse_iter": best_cfg[0],
        "outlier_fraction": best_cfg[1],
        "f_score": watershed_scores[best_idx][0],
        "map": watershed_scores[best_idx][1]
    }

    print("\n--- BEST CONFIGURATION (DRLSE + SMART) ---")
    print(best_res)

    param_file_path = os.path.join(RESULT_DIR, 'best_watershed_params.json')
    with open(param_file_path, 'w') as f:
        json.dump(best_res, f, indent=4)
    print(f"✅ Saved to: {param_file_path}")

Evaluating DRLSE Results...
Iter=0, OutFrac=0.05 | F1: 0.0014, mAP: 0.0002
Iter=0, OutFrac=0.1 | F1: 0.0014, mAP: 0.0002
Iter=0, OutFrac=0.2 | F1: 0.0014, mAP: 0.0002
Iter=10, OutFrac=0.05 | F1: 0.0014, mAP: 0.0002
Iter=10, OutFrac=0.1 | F1: 0.0014, mAP: 0.0002
Iter=10, OutFrac=0.2 | F1: 0.0014, mAP: 0.0002
Iter=30, OutFrac=0.05 | F1: 0.0014, mAP: 0.0002
Iter=30, OutFrac=0.1 | F1: 0.0014, mAP: 0.0002
Iter=30, OutFrac=0.2 | F1: 0.0014, mAP: 0.0002

--- BEST CONFIGURATION (DRLSE + SMART) ---
{'method': 'drlse_smart_filter', 'drlse_iter': 0, 'outlier_fraction': 0.05, 'f_score': 0.0014009526458378361, 'map': 0.00020607933402061462}
✅ Saved to: /content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_with_denoise_reference_CRF/10017/unet_eb5_dice_CRF/best_watershed_params.json


## Ending

### Write the best hyperparameters

In [ ]:
if F_score:
    cv_scores_sorted = sorted(cv_scores, key=lambda x: x[0], reverse=True)
    csorted_indices = sorted(range(len(cv_scores)), key=lambda i: cv_scores[i][0], reverse=True)
else:
    cv_scores_sorted = sorted(cv_scores, key=lambda x: x[1], reverse=True)
    csorted_indices = sorted(range(len(cv_scores)), key=lambda i: cv_scores[i][1], reverse=True)
cv_scores_sorted[0], cv_config[csorted_indices[0]]

((0.9100159083476289, 0.7127978801727295), (4, 0.6))

In [ ]:
if F_score:
    tp_scores_sorted = sorted(tp_scores, key=lambda x: x[0], reverse=True)
    tsorted_indices = sorted(range(len(tp_scores)), key=lambda i: tp_scores[i][0], reverse=True)
else:
    tp_scores_sorted = sorted(tp_scores, key=lambda x: x[1], reverse=True)
    tsorted_indices = sorted(range(len(tp_scores)), key=lambda i: tp_scores[i][1], reverse=True)
tp_scores_sorted[0], tp_config[tsorted_indices[0]]

((0.9234351378038682, 0.7275786995887756), (0.25, 0.4))

In [ ]:
if F_score:
    nms_scores_sorted = sorted(nms_scores, key=lambda x: x[0], reverse=True)
    nsorted_indices = sorted(range(len(nms_scores)), key=lambda i: nms_scores[i][0], reverse=True)
else:
    nms_scores_sorted = sorted(nms_scores, key=lambda x: x[1], reverse=True)
    nsorted_indices = sorted(range(len(nms_scores)), key=lambda i: nms_scores[i][1], reverse=True)
nms_scores_sorted[0], nms_config[nsorted_indices[0]]

((0.951025750335122, 0.7358970642089844), (0.5, 0.6))

In [ ]:
with open("best_config.txt", "w") as f:
    f.write(f"cv_config: {cv_config[csorted_indices[0]]}\n")
    f.write(f"tp_config: {tp_config[tsorted_indices[0]]}\n")
    f.write(f"nms_config: {nms_config[nsorted_indices[0]]}\n")

In [ ]:
!cp best_config.txt {RESULT_DIR}

In [ ]:
# @markdown ---
# @markdown time used
end_time = time.time()
print(f"End time recorded: {end_time}")

elapsed_time = end_time - start_time
elapsed_time = elapsed_time


hours = int(elapsed_time // 3600)
remaining_seconds = elapsed_time % 3600

minutes = int(remaining_seconds // 60)
seconds = round(remaining_seconds % 60, 3)

print(f"Time spend : {hours} h, {minutes} m, {seconds} s")


gpu_used = "L4" # @param ["CPU high", "T4", "T4 high", "L4"]
per_unit_cost_dict = {"L4" : 1.71, "T4 high" : 1.41, "T4" : 1.19, "CPU high" :  0.24}
per_unit_cost = per_unit_cost_dict[gpu_used]
print(f"unit price per hr {per_unit_cost}")

cost_units = per_unit_cost * elapsed_time / 3600

per_unit_US = 10.49 / 100

cost_price_US = cost_units * per_unit_US

print(f"unit cost : {round(cost_units, 4)}")
print(f"unit price US: {cost_price_US}")
print(f"unit price NTD: {cost_price_US * 30.76}")

End time recorded: 1765271779.3454504
Time spend : 910 h, 22 m, 25.402 s
unit price per hr 1.71
unit cost : 1556.7391
unit price US: 163.30192801031185
unit price NTD: 5023.167305597193
